# RunModel27.2-province-demo: Thailand Malaria Forecasting, Province-Level Public-Data Demo (2016-2026)

> RunModel27.2-province-demo = RunModel27.1-parallel (PIPELINE_CODE_VERSION 5) run on **aggregated province-month** data so that the full pipeline can be tried without access to the restricted case-level surveillance records. The study design (2016-2025, five expanding-window folds, 2025 external test, 2026 forecast) and the modelling code (Sections B, Q and S) are unchanged. Results go to `outputs_model27_2_province_demo`.

## What differs from RunModel27.1-parallel

| Item | RunModel27.1-parallel | This demo |
| --- | --- | --- |
| Malaria input | `Malaria_10Y.csv`, one row per case (restricted) | `malaria_province_monthly_2559_2568.csv`, monthly malaria counts by province from the DDC Malaria Online system; `download_malaria_monthly_2568.py 2016 2025` downloads the same table from the public DDC malaria MIS overview |
| Case count | Number of case records | `All_All`: Thai + M1 + M2 migrants, all species |
| Climate input | `ERA5Land_Thailand_2016_2025.csv` | Same file |
| Spatial unit | 811 districts | 77 provinces; climate is the unweighted mean over the districts of each province |
| Person characteristics | Sex and age tables | Not available (aggregated input) |

**Column names.** To keep the modelling code identical, the unit column keeps the name `ADM2_CODE` and the word "district" is kept in code, file names and figure titles. In this notebook `ADM2_CODE` holds the **2-digit province code** (DOPA district code // 100) and "district" means province.

**Paths.** Data files are read from, and outputs written to, the directory in the `MALARIA_PROJECT_DIR` environment variable, or the directory the notebook runs from when the variable is not set. The `MALARIA_FILE` environment variable, when set, replaces the default malaria file path.

> Demo only: province-level results show that the tool runs end to end. They are not the district-level results reported in the manuscript.

> Important: the GeoAI package supports geospatial processing and visualization; it does not replace or alter the provenance of the malaria, climate, or administrative-boundary data used in this study.


In [ ]:
# ================================================================
# PARALLEL DISTRICT WORKERS AND COMPUTE DEVICE
# Must run before TensorFlow or any math library is imported, hence this
# being the first cell. Restart the kernel after changing PARALLEL_WORKERS.
#
# PARALLEL_WORKERS = 1 : Sections Q and S run one district at a time in
#                        this kernel; TensorFlow and XGBoost use the GPU.
# PARALLEL_WORKERS > 1 : Sections Q and S run that many districts at once
#                        in forked worker processes, on the CPU only.
#
# Why RunModel27 (v5) deadlocked with PARALLEL_WORKERS > 1, and the fix:
#   RunModel27 imported TensorFlow in this kernel (cell 1 GPU probe, cell 4
#   Keras imports and tf.random.set_seed, cell 17 class definitions) long
#   before Section Q forked its pool. TensorFlow's C++ runtime starts
#   background threads and takes internal locks at import and first use.
#   fork() copies only the calling thread, so each child inherited locks
#   whose owning threads no longer existed: 5 of 6 workers sat at 0% CPU in
#   futex_wait_queue for 17+ minutes (verified 2026-09-25).
#   Here, in parallel mode, this kernel never imports TensorFlow before the
#   pools of Sections Q and S have been forked. Every TensorFlow use is
#   inside a function that runs in a worker, and each worker imports
#   TensorFlow itself after fork -- the structure RunModel25 uses.
#   run_district_tasks() refuses to fork if TensorFlow is already loaded,
#   so a future edit that imports it early fails loudly instead of hanging.
#
# Other fork rules (also from RunModel25):
#   - CUDA does not survive fork, so the GPU is hidden and workers use the
#     CPU. XGBoost follows NEURAL_DEVICE.
#   - Every math library is limited to one thread per process, set here
#     before numpy/sklearn/xgboost load: a library that has already started
#     a thread pool leaves its locks behind at fork. One thread per worker
#     also matches a pool sized to the CPU cores.
#
# NEURAL_DEVICE is part of the run signature (GPU and CPU fits differ
# numerically), so switching between 1 and more workers recomputes rather
# than mixing checkpoints. The worker count itself is not in the signature:
# on the same device any worker count gives the same results, because every
# district reseeds from its own ADM2_CODE and writes its own checkpoint.
# ================================================================
PARALLEL_WORKERS = 8
# 8 = every vCPU WSL exposes (.wslconfig processors=8). Measured with 6
# workers: each ~97% of one CPU, ~1.7 GB RSS, VM total ~7 GB, ~24% CPU idle.
# Tested 2026-09-25 in a real nbconvert/Jupyter kernel (4 high-burden
# districts): no deadlock; 2 workers finished Section Q in 3.0 min and 3
# workers in 2.7 min, and both gave byte-identical checkpoints (results,
# MC-dropout intervals, 2025 test, 2026 forecast). About 1.5 CPU-minutes per
# district per worker. Each worker lives only ~2 districts, so it never
# reaches RunModel25's 4.8 GB; with 2 workers the VM used ~2-3 GB more.
# Watch `free -g` in WSL on a first full run anyway.
# A worker process is replaced after this many districts. TensorFlow never
# returns its heap to the OS, so ending the process is the only way to free
# that memory.
WORKER_MAX_DISTRICTS = 2

import os
import sys

if PARALLEL_WORKERS > 1:
    if "tensorflow" in sys.modules:
        raise RuntimeError(
            "TensorFlow is already loaded in this kernel, so forked workers would "
            "deadlock. Restart the kernel and run the notebook from this cell."
        )
    for _variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                      "NUMEXPR_NUM_THREADS", "TF_NUM_INTRAOP_THREADS", "TF_NUM_INTEROP_THREADS"):
        os.environ.setdefault(_variable, "1")
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")
    # TensorFlow is deliberately NOT imported here; see above.
    NEURAL_DEVICE = "cpu"
else:
    # ============================================================
    # CUDA PTX JIT CACHE SIZE (GPU mode only)
    # If this TF build has no prebuilt kernel binaries for the GPU's
    # compute capability, the CUDA driver JIT-compiles them from PTX on
    # first use. The driver's default JIT cache is only 256 MB, which this
    # workload overflows; once it thrashes, kernels are recompiled on every
    # run and each JIT-compiled module stays resident in host RAM. A large
    # cache pays the compile cost once. setdefault: a shell setting wins.
    # This only changes caching; it cannot change any computed value.
    # ============================================================
    os.environ.setdefault("CUDA_CACHE_MAXSIZE", "4294967296")  # 4 GB
    import tensorflow as tf
    NEURAL_DEVICE = "gpu" if tf.config.list_physical_devices("GPU") else "cpu"

print("PARALLEL_WORKERS:", PARALLEL_WORKERS, "| NEURAL_DEVICE:", NEURAL_DEVICE)


In [ ]:
# ================================================================
# Thailand Malaria Forecasting Workflow (2016-2025, province-level public-data demo)
# Process-aligned version based on RunModel11 and the attached workflow figure

In [ ]:
# ================================================================
# Purpose:
# This script restructures the full analysis so that it follows the workflow:
# 1) Data collection
# 2) Data integration
# 3) Data preparation & preprocessing
# 4) Forecasting models
# 5) Model validation
# 6) Identify risk districts
#
# Design principles:
# - Preserve true zero-case months
# - Use monthly district-level panel data
# - Add lag features (1, 2, 3 months)
# - Scale predictors with Min-Max normalization
# - Use the full climate predictor set defined by the conceptual workflow
# - Manage outliers district-by-district
# - Compare classical time-series models and deep learning models
# - Select the best model using rolling-origin validation
# - Classify 2026 risk using the most recent 5-year historical median
# ================================================================

In [ ]:
# ================================================================
# PACKAGE INSTALLATION AND IMPORTS
# ================================================================
!pip -q install openpyxl pmdarima xgboost geopandas

import warnings
warnings.filterwarnings("ignore")

import os
import gc
import json
import math
import random
import subprocess
import sys
import time
import hashlib
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor

# TensorFlow / Keras are imported inside the functions that use them (cell
# 17 and Sections Q and S), never here: in parallel mode this kernel must
# not load TensorFlow before the worker pools are forked (see cell 1).
# tf.random.set_seed(SEED) is gone for the same reason; it never mattered,
# because every district reseeds itself (seed_district) before any neural fit.

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## STUDY METHODOLOGY AND ANALYTICAL DESIGN

The unit of analysis is the province-month. Monthly malaria counts and environmental predictors are integrated for 2016–2025, processed without replacing true zero case counts, and used to forecast monthly malaria cases in 2026. Five expanding-window folds use validation years 2020–2024. The year 2025 is an external test used only after model selection; the 2026 forecast refits on data through 2025. Seven candidate models are compared using MAE, RMSE, and sMAPE on complete, matching folds within each province. The lowest mean validation RMSE is the primary model-selection criterion, followed by MAE and sMAPE as tie-breakers.

| Stage | Training years (CE) | Evaluation/forecast year (CE) | Horizon |
| --- | --- | --- | --- |
| Fold 1 | 2016–2019 | 2020 | 12 months |
| Fold 2 | 2016–2020 | 2021 | 12 months |
| Fold 3 | 2016–2021 | 2022 | 12 months |
| Fold 4 | 2016–2022 | 2023 | 12 months |
| Fold 5 | 2016–2023 | 2024 | 12 months |
| External test | 2016–2024 | 2025 | 12 months |
| Final forecast | 2016–2025 | 2026 | 12 months |

The first three months of 2016 initialize lag features and are excluded from model rows, leaving 45 training months for Fold 1.

### Resource-conscious KNN parameter validation

KNN imputation is evaluated at `k = 3, 5, 7, 9, 11`. Validation uses only training-period complete predictor records, a fixed random seed, a maximum of 2,000 sampled rows, and 10% artificial masking. Predictors are standardized within the validation experiment because KNN distance is scale-sensitive. The value of `k` with the lowest mean standardized RMSE is selected; the smaller `k` is preferred if performance is tied. This validation does not modify malaria outcomes and does not use the 2025 external test outcome.

### Seven-model comparison

- ARIMA: univariate non-seasonal statistical model.
- SARIMA: univariate seasonal statistical model.
- SARIMAX: seasonal statistical model with environmental predictors.
- Univariate LSTM: recurrent neural network using the malaria series.
- Multivariate LSTM: recurrent neural network using malaria and engineered predictors.
- GRU: compact multivariate recurrent neural network with fewer recurrent gates than LSTM.
- XGBoost: regularized gradient-boosted trees using recursively updated outcome lags and environmental predictors.

The neural networks use time-ordered internal validation, early stopping, and no random shuffling. XGBoost uses fixed, regularized parameters to avoid a costly district-by-district hyperparameter search. Forecasts are constrained to non-negative values because the outcome is a disease count.


## GEOAI EXTENSION AND DATA PROVENANCE

The optional spatial extension references the official [OpenGeoAI documentation](https://opengeoai.org/) and the [GeoAI source repository](https://github.com/opengeos/geoai). The documented `geoai-py` package integrates AI with raster and vector geospatial workflows, including data discovery/download, dataset preparation, inference, and visualization.

For this study, GeoAI is used only as an optional spatial-processing and visualization layer. Epidemiologic case data, ERA5-Land climate variables, and Thailand district boundaries must retain their original source citations. The core district-level vector join uses GeoPandas so that tabular forecasting remains reproducible even when the heavier GeoAI dependencies are not installed.


In [ ]:
# ================================================================
# OPTIONAL OPENGEOAI PACKAGE SETUP
# Official documentation: https://opengeoai.org/
# PyPI installation name: geoai-py; Python import name: geoai
# ================================================================
INSTALL_OPEN_GEOAI = False

if INSTALL_OPEN_GEOAI:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "geoai-py"])

try:
    import geoai
    GEOAI_AVAILABLE = True
    print("OpenGeoAI package is available.")
except ImportError:
    geoai = None
    GEOAI_AVAILABLE = False
    print("OpenGeoAI optional extension is disabled. Set INSTALL_OPEN_GEOAI=True and rerun this cell to enable it.")


## SECTION A. DATA COLLECTION: FILE PATHS AND GLOBAL SETTINGS
Explanation:
- MALARIA_FILE: monthly province-level malaria counts (public DDC malaria MIS overview)
- CLIMATE_FILE: monthly district-level climate data, averaged to province in Section F
- The historical period is 2016-2025
- 2025 is reserved for external testing
- 2026 is forecasted using proxy climate from 2025 unless real 2026 climate is available

In [ ]:
# ================================================================
# SECTION A. DATA COLLECTION: FILE PATHS AND GLOBAL SETTINGS
# Explanation:
# - MALARIA_FILE: monthly province-level malaria counts (public DDC malaria MIS overview)
# - CLIMATE_FILE: monthly district-level climate data, averaged to province in Section F
# - The historical period is 2016-2025
# - 2025 is reserved for external testing
# - 2026 is forecasted using proxy climate from 2025 unless real 2026 climate is available

In [ ]:
# ================================================================
# Data files are read from, and outputs written to, PROJECT_DIR: the
# MALARIA_PROJECT_DIR environment variable when it is set, otherwise the
# directory the notebook runs from ("."), so printed paths stay relative.
# MALARIA_FILE, when set, replaces the default malaria file path.
PROJECT_DIR = os.environ.get("MALARIA_PROJECT_DIR", ".")

MALARIA_FILE = os.environ.get(
    "MALARIA_FILE", os.path.join(PROJECT_DIR, "malaria_province_monthly_2559_2568.csv")
)
CLIMATE_FILE = os.path.join(PROJECT_DIR, "ERA5Land_Thailand_2016_2025.csv")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs_model27_2_province_demo")

HIST_START_YEAR = 2016
HIST_END_YEAR = 2025
TRAIN_END_YEAR = 2024
TEST_YEAR = 2025
FORECAST_YEAR = 2026
RECENT_MEDIAN_START = 2021  # 5-year window: 2021-2025

ROLLING_FOLDS = 5
TEST_HORIZON = 12
LOOK_BACK = 12
FIRST_VALIDATION_YEAR = 2020
MIN_FOLD_TRAIN_MONTHS = 45  # Jan-Mar 2016 are warm-up rows for lag features
EXPANDING_VALIDATION_YEARS = list(range(FIRST_VALIDATION_YEAR, TEST_YEAR))
if len(EXPANDING_VALIDATION_YEARS) != ROLLING_FOLDS:
    raise ValueError("The number of validation years must equal ROLLING_FOLDS.")
FAST_MODE = True
MAX_DISTRICTS = None   # set an integer for quick pilot runs (provinces here)
USE_DEEP_LEARNING = True

# Bump this by 1 whenever the MODELLING CODE changes in a way that can
# alter a district's numbers -- e.g. editing the model builders, the
# training helpers, or the recursive-forecast functions in Section B.
# It is folded into the run signature, so bumping it invalidates every
# existing per-district checkpoint and forces a clean recomputation.
# Without this, the signature only tracks PARAMETERS, so a code edit
# would leave old checkpoints looking current and the final CSVs would
# silently mix results produced by two different versions of the code.
#
# History:
#   1 = original .predict()-based recursive forecasting
#   2 = recursive forecasting switched to direct eager model calls
#   3 = FAST_MODE extended to the LSTM search: under FAST_MODE the LSTM
#       uses the single locked specification instead of a 4-candidate grid
#   4 = five 2020-2024 folds, independent 2025 test, complete-fold selection
#   5 = Multivariate LSTM/GRU recursive forecasts rebuild the malaria lag and
#       rolling-mean predictors from their own predictions (validation, 2025
#       test and 2026 alike); MC-dropout 95% intervals via an MCDropout layer;
#       ARIMA/SARIMA/SARIMAX interval bounds clipped at 0; per-district seeding
#       via keras.utils.set_random_seed
PIPELINE_CODE_VERSION = 5

# Per-district random seeding for the neural models (Section Q and S).
#   True  (v5 default): keras.utils.set_random_seed(SEED + district_id).
#         Tested in this environment (TF 2.21, Keras 3.14): two runs with
#         the same seed give identical weights, dropout masks and
#         MC-dropout intervals. It also reseeds Python `random` and NumPy.
#   False (v4 behaviour): tf.random.set_seed(SEED + district_id).
#         Under Keras 3 this does NOT drive weight initialisation or dropout
#         masks, so two runs with the same seed give different neural
#         results and different MC-dropout intervals.
# This flag is part of the run signature, so switching it forces
# recomputation instead of mixing checkpoints from both settings.
USE_KERAS_RANDOM_SEED = True

os.makedirs(OUTPUT_DIR, exist_ok=True)
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress_model27_2_province_demo.json")
DISTRICT_REPORTS_DIR = os.path.join(OUTPUT_DIR, "district_reports")
os.makedirs(DISTRICT_REPORTS_DIR, exist_ok=True)
FORECAST_REPORTS_DIR = os.path.join(OUTPUT_DIR, "forecast_2026_reports")
os.makedirs(FORECAST_REPORTS_DIR, exist_ok=True)
SECTION_CHECKPOINT_LOG = os.path.join(OUTPUT_DIR, "section_checkpoints.jsonl")

def save_progress(stage, completed_districts=0, district_id=None, fold_id=None):
    payload = {
        "stage": stage,
        "completed_districts": int(completed_districts),
        "district_id": None if district_id is None else str(district_id),
        "fold_id": None if fold_id is None else str(fold_id),
    }
    # Worker processes write this file too, so write a per-process temp
    # file and rename it; two writers can then never interleave their output.
    tmp_path = f"{PROGRESS_FILE}.{os.getpid()}.tmp"
    with open(tmp_path, "w", encoding="utf-8") as progress_file:
        json.dump(payload, progress_file, ensure_ascii=False)
    os.replace(tmp_path, PROGRESS_FILE)

print("Historical window:", HIST_START_YEAR, "-", HIST_END_YEAR)
print("Train end year:", TRAIN_END_YEAR)
print("Test year:", TEST_YEAR)
print("Forecast year:", FORECAST_YEAR)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# ================================================================
# MEMORY-SAFE CHECKPOINTING HELPERS
# Defined here (right after Section A) so every later section can use
# them. These only change *when* intermediate data is written to disk;
# they do not alter any computed value, so all numeric results stay
# identical to the original notebook.
# ================================================================
def to_jsonable(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        v = float(value)
        return None if np.isnan(v) else v
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).isoformat()
    if isinstance(value, float) and np.isnan(value):
        return None
    return value


def row_to_jsonable(row):
    return [to_jsonable(v) for v in row]


def mark_section_checkpoint(section_name, **info):
    record = {
        "section": section_name,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    record.update({k: to_jsonable(v) for k, v in info.items()})
    with open(SECTION_CHECKPOINT_LOG, "a", encoding="utf-8") as log_file:
        log_file.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"[checkpoint] {section_name} done at {record['timestamp']}")


def compute_run_signature():
    # Fingerprints what determines per-district output, so a district JSON
    # produced under an OLDER setup is never silently reused after that
    # setup changes. Two kinds of change are covered:
    #   - PARAMETERS: feature set, KNN k, lag window, validation years,
    #     XGBoost params, model on/off flags.
    #   - CODE: via PIPELINE_CODE_VERSION, which the author bumps after
    #     editing the modelling code itself. Parameters alone cannot
    #     detect a code edit, and an undetected one would let the final
    #     CSVs mix results from two different versions of the code.
    payload = {
        "pipeline_code_version": int(PIPELINE_CODE_VERSION),
        # FAST_MODE changes the ARIMA/SARIMA/SARIMAX grids AND (from code
        # version 3) the LSTM search, so toggling it must invalidate every
        # existing checkpoint. It was missing from this payload before.
        "fast_mode": bool(FAST_MODE),
        "train_end_year": TRAIN_END_YEAR,
        "test_year": TEST_YEAR,
        "look_back": LOOK_BACK,
        "expanding_validation_years": list(EXPANDING_VALIDATION_YEARS),
        "min_fold_train_months": MIN_FOLD_TRAIN_MONTHS,
        "feature_cols": list(feature_cols),
        "selected_knn_k": int(SELECTED_KNN_K),
        "xgb_params": {k: v for k, v in XGB_PARAMS.items()},
        "run_gru": RUN_GRU,
        "run_xgboost": RUN_XGBOOST,
        "use_deep_learning": USE_DEEP_LEARNING,
        "mc_dropout_iter": int(MC_DROPOUT_ITER),
        "use_keras_random_seed": bool(USE_KERAS_RANDOM_SEED),
        # GPU and CPU give numerically different fits. PARALLEL_WORKERS > 1
        # forces the CPU, so this keeps the two kinds of checkpoint apart.
        "neural_device": NEURAL_DEVICE,
    }
    signature_text = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.sha256(signature_text.encode("utf-8")).hexdigest()


def report_is_current(district_id, run_signature):
    # A district's checkpoint only counts as "done" if it exists, parses,
    # AND matches the CURRENT run's signature. A partially-written file
    # (e.g. process killed mid-write) or one left over from a different
    # configuration is treated as not done, so it gets recomputed instead
    # of silently reused.
    report_path = os.path.join(DISTRICT_REPORTS_DIR, f"{int(district_id)}.json")
    if not os.path.exists(report_path):
        return False
    try:
        with open(report_path, encoding="utf-8") as report_file:
            payload = json.load(report_file)
    except (json.JSONDecodeError, OSError):
        return False
    return payload.get("run_signature") == run_signature


def save_district_report(district_id, results_rows, interval_rows_local, forecast_rows_local,
                          parameter_rows_local, neural_loss_rows_local, run_signature):
    report_path = os.path.join(DISTRICT_REPORTS_DIR, f"{int(district_id)}.json")
    tmp_path = report_path + ".tmp"
    payload = {
        "run_signature": run_signature,
        "results": [row_to_jsonable(r) for r in results_rows],
        "interval": [row_to_jsonable(r) for r in interval_rows_local],
        "forecast": [row_to_jsonable(r) for r in forecast_rows_local],
        "parameters": [row_to_jsonable(r) for r in parameter_rows_local],
        "neural_loss_history": [row_to_jsonable(r) for r in neural_loss_rows_local],
    }
    # Write to a temp file and rename atomically so a process kill mid-write
    # (the exact failure mode this checkpointing exists to survive) can never
    # leave behind a truncated file that report_is_current() mistakes for done.
    with open(tmp_path, "w", encoding="utf-8") as report_file:
        json.dump(payload, report_file, ensure_ascii=False)
    os.replace(tmp_path, report_path)


def load_district_reports(district_ids, run_signature):
    all_reports = []
    for district_id in district_ids:
        report_path = os.path.join(DISTRICT_REPORTS_DIR, f"{int(district_id)}.json")
        if not os.path.exists(report_path):
            continue
        try:
            with open(report_path, encoding="utf-8") as report_file:
                payload = json.load(report_file)
        except (json.JSONDecodeError, OSError):
            continue
        if payload.get("run_signature") == run_signature:
            all_reports.append(payload)
    return all_reports


# Section S (2026 forecast) has the same per-district neural-model-building
# shape as Section Q, so it gets the identical checkpoint treatment: one
# JSON per district, atomic write, signature-gated resume.
def forecast_report_is_current(district_id, run_signature):
    report_path = os.path.join(FORECAST_REPORTS_DIR, f"{int(district_id)}.json")
    if not os.path.exists(report_path):
        return False
    try:
        with open(report_path, encoding="utf-8") as report_file:
            payload = json.load(report_file)
    except (json.JSONDecodeError, OSError):
        return False
    return payload.get("run_signature") == run_signature


def save_forecast_report(district_id, forecast_rows_local, run_signature):
    report_path = os.path.join(FORECAST_REPORTS_DIR, f"{int(district_id)}.json")
    tmp_path = report_path + ".tmp"
    payload = {
        "run_signature": run_signature,
        "forecast": [row_to_jsonable(r) for r in forecast_rows_local],
    }
    with open(tmp_path, "w", encoding="utf-8") as report_file:
        json.dump(payload, report_file, ensure_ascii=False)
    os.replace(tmp_path, report_path)


def load_forecast_reports(district_ids, run_signature):
    all_reports = []
    for district_id in district_ids:
        report_path = os.path.join(FORECAST_REPORTS_DIR, f"{int(district_id)}.json")
        if not os.path.exists(report_path):
            continue
        try:
            with open(report_path, encoding="utf-8") as report_file:
                payload = json.load(report_file)
        except (json.JSONDecodeError, OSError):
            continue
        if payload.get("run_signature") == run_signature:
            all_reports.append(payload)
    return all_reports


# ================================================================
# DISTRICT TASK RUNNER: SERIAL OR FORKED WORKER POOL
# ================================================================
_DISTRICT_TASK = None


def _run_district_task(item):
    # Module-level so the pool can reference it by name. The task function
    # itself is inherited through fork and never pickled.
    _DISTRICT_TASK(item)
    return item


def run_district_tasks(task, items, label, report_every=10):
    """Run task(item) for every item: serially, or in PARALLEL_WORKERS forked processes.

    Each task writes its own checkpoint file named by ADM2_CODE and reseeds
    from that code, so the order in which workers finish cannot change any
    result. Forked workers inherit model_df and every helper from this
    kernel. A worker is replaced after WORKER_MAX_DISTRICTS districts to
    release TensorFlow memory. An exception in any task stops the run, as
    it would in the serial loop; finished districts stay checkpointed.
    """
    global _DISTRICT_TASK
    items = list(items)
    total = len(items)
    if total == 0:
        print(f"{label}: every district checkpoint is already current; nothing to run.")
        return
    workers = min(PARALLEL_WORKERS, total)
    started = time.time()
    print(f"{label}: {total} districts to run | workers {workers} | device {NEURAL_DEVICE}", flush=True)

    def progress(done, item):
        if done % report_every == 0 or done == total:
            district = item[0] if isinstance(item, tuple) else item
            minutes = (time.time() - started) / 60
            print(f"{label}: {done}/{total} done (last {district}) | {minutes:.1f} min", flush=True)

    if workers <= 1:
        for done, item in enumerate(items, start=1):
            task(item)
            progress(done, item)
        return

    import multiprocessing
    # The fork-safety rule of cell 1, enforced: forking after TensorFlow has
    # loaded in this kernel is what deadlocked RunModel27 v5.
    if "tensorflow" in sys.modules:
        raise RuntimeError(
            "TensorFlow was imported in this kernel before the worker pool was forked, "
            "so the workers would deadlock. Find the cell that imports it outside a "
            "worker function, restart the kernel, and rerun."
        )
    if "fork" not in multiprocessing.get_all_start_methods():
        raise RuntimeError("PARALLEL_WORKERS > 1 needs the fork start method (Linux/WSL kernel). "
                           "Set PARALLEL_WORKERS = 1 on Windows.")
    _DISTRICT_TASK = task
    try:
        with multiprocessing.get_context("fork").Pool(
            workers, maxtasksperchild=WORKER_MAX_DISTRICTS
        ) as pool:
            for done, item in enumerate(pool.imap_unordered(_run_district_task, items), start=1):
                progress(done, item)
    finally:
        _DISTRICT_TASK = None


In [ ]:
mark_section_checkpoint("A_setup")

## SECTION A1. STUDY PARAMETERS AND MODEL SETTINGS

This section explicitly records all parameters used in the study so that the analysis is reproducible and can be reported in the methodology chapter.

In [ ]:
# ================================================================
# SECTION A1. STUDY PARAMETERS AND MODEL SETTINGS
# ================================================================
KNN_CANDIDATES = [3, 5, 7, 9, 11]
KNN_VALIDATION_MAX_ROWS = 2000
KNN_MASK_FRACTION = 0.10
KNN_VALIDATION_REPEATS = 1

RUN_GRU = True
RUN_XGBOOST = True
GRU_EPOCHS = 75
GRU_BATCH_SIZE = 16
GRU_PATIENCE = 10

# MC-dropout 95% interval for Univariate LSTM, Multivariate LSTM and GRU:
# number of stochastic forecast paths; the interval is their 2.5th-97.5th
# percentile band.
MC_DROPOUT_ITER = 30 if FAST_MODE else 100

XGB_PARAMS = {
    "n_estimators": 200,
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "min_child_weight": 2,
    "reg_alpha": 0.05,
    "reg_lambda": 1.0,
    "objective": "reg:squarederror",
    "random_state": SEED,
    "n_jobs": 2,
    # CPU whenever the GPU is hidden for PARALLEL_WORKERS > 1 (see cell 1).
    "device": "cuda" if NEURAL_DEVICE == "gpu" else "cpu",
}

study_parameters = pd.DataFrame([
    ["Historical study period", f"{HIST_START_YEAR}-{HIST_END_YEAR}"],
    ["Training period", f"{HIST_START_YEAR}-{TRAIN_END_YEAR}"],
    ["External test year", TEST_YEAR],
    ["Forecast year", FORECAST_YEAR],
    ["Recent median risk baseline", f"{RECENT_MEDIAN_START}-{HIST_END_YEAR}"],
    ["Expanding validation years", str(EXPANDING_VALIDATION_YEARS)],
    ["Forecast horizon per fold", TEST_HORIZON],
    ["Lag / look-back value", LOOK_BACK],
    ["KNN candidate values", str(KNN_CANDIDATES)],
    ["KNN validation sample cap", KNN_VALIDATION_MAX_ROWS],
    ["KNN artificial masking fraction", KNN_MASK_FRACTION],
    ["Outlier method", "District-level IQR capping after K-means grouping"],
    ["Clustering method", "KMeans, n_clusters=4, random_state=42, n_init=10"],
    ["Normalization", "MinMaxScaler fitted on training data only"],
    ["ARIMA search grid in FAST_MODE", "(0,1,1), (1,1,1), (1,0,1)"],
    ["SARIMA/SARIMAX seasonal grid in FAST_MODE", "(0,0,0,0), (1,1,0,12), (1,1,1,12)"],
    ["Classical model selection", "Lowest AIC among candidate specifications"],
    ["LSTM tuning", (
        "Single locked regularized specification (64/32 units, dropout 0.20, "
        "lr 0.001, Huber loss) with time-based validation and early stopping"
        if FAST_MODE else
        "Regularized candidate search over 4 specifications with time-based "
        "validation and early stopping"
    )],
    ["Neural search mode", f"FAST_MODE={FAST_MODE}"],
    ["GRU tuning", "Single compact regularized specification with early stopping"],
    ["XGBoost tuning", "Fixed regularized specification to limit computational burden"],
    ["Model comparison", "ARIMA, SARIMA, SARIMAX, Univariate LSTM, Multivariate LSTM, GRU, XGBoost"],
    ["Validation indicators", "MAE, RMSE, sMAPE"],
    ["Primary model-selection rule", "Lowest mean expanding-window RMSE; MAE and sMAPE as tie-breakers"],
    ["Uncertainty level", "95% confidence or prediction interval where supported"],
    ["Neural-network interval", f"MC dropout, {MC_DROPOUT_ITER} forecast paths, 2.5th-97.5th percentiles"],
    ["Optional spatial package", "geoai-py from https://opengeoai.org/"],
], columns=["Parameter", "Value"])

display(study_parameters)
study_parameters.to_csv(f"{OUTPUT_DIR}/study_parameters.csv", index=False)
mark_section_checkpoint("A1_study_parameters")

## SECTION B. HELPER FUNCTIONS

In [ ]:
# ================================================================
# SECTION B. HELPER FUNCTIONS

In [ ]:
# ================================================================
# SECTION B. HELPER FUNCTIONS
# ================================================================
def first_existing(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of the candidate columns were found: {candidates}")
    return None

def safe_numeric_frame(df, cols, fill_value=0.0):
    out = df.copy()
    for c in cols:
        if c not in out.columns:
            out[c] = fill_value
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out[cols] = out[cols].fillna(fill_value).astype(float)
    return out


def ensure_columns(df, cols, fill_value=np.nan):
    out = df.copy()
    for c in cols:
        if c not in out.columns:
            out[c] = fill_value
    return out


def dedup_keep_order(seq):
    return list(dict.fromkeys(seq))


def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    out = np.where(denom == 0, 0, np.abs(y_true - y_pred) / denom)
    return np.mean(out) * 100


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "sMAPE": smape(y_true, y_pred),
    }


def build_month_index(start_year, end_year):
    rows = [(y, m) for y in range(start_year, end_year + 1) for m in range(1, 13)]
    return pd.DataFrame(rows, columns=["year", "month"])


def add_date_columns(df):
    df = df.copy()
    df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
    return df.sort_values(["ADM2_CODE", "date"]).reset_index(drop=True)


def create_lag_features(df, group_col, target_cols, lags=(1, 2, 3)):
    df = df.copy()
    for col in target_cols:
        for lag in lags:
            df[f"{col}_lag{lag}"] = df.groupby(group_col)[col].shift(lag)
    return df


def add_cyclical_month_features(df):
    df = df.copy()
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df


def add_rolling_features(df, group_col, target_col, windows=(3, 6)):
    df = df.copy()
    for w in windows:
        df[f"{target_col}_rollmean_{w}"] = (
            df.groupby(group_col)[target_col]
              .shift(1)
              .rolling(w)
              .mean()
              .reset_index(level=0, drop=True)
        )
    return df



def build_expanding_yearly_folds(df, start_year=2016, validation_years=None):
    if validation_years is None:
        validation_years = EXPANDING_VALIDATION_YEARS

    folds = []
    for validation_year in validation_years:
        if validation_year >= TEST_YEAR:
            raise ValueError("External test year must not be used for model selection.")
        train_end_year = validation_year - 1
        train_mask = (df["year"] >= start_year) & (df["year"] <= train_end_year)
        test_mask = df["year"] == validation_year

        if train_mask.sum() == 0 or test_mask.sum() == 0:
            continue

        folds.append({
            "train_start_year": start_year,
            "train_end_year": train_end_year,
            "validation_year": validation_year,
        })

    return folds


def select_complete_validation_results(results, validation_years):
    # Compare each district's models on the same complete set of folds.
    required = set(validation_years)
    metric_cols = ["MAE", "RMSE", "sMAPE"]
    return results.groupby(["ADM2_CODE", "model"], group_keys=False).filter(
        lambda group: len(group) == len(required)
        and set(group["fold"]) == required
        and np.isfinite(group[metric_cols].to_numpy(dtype=float)).all()
    )


def search_sarimax(train_y, exog_train=None, seasonal=False, fast_mode=True):
    if fast_mode:
        orders = [(0,1,1), (1,1,1), (1,0,1)]
        seas = [(0,0,0,0), (1,1,0,12), (1,1,1,12)] if seasonal else [(0,0,0,0)]
    else:
        orders = list(product([0,1,2], [0,1], [0,1,2]))
        seas = [(0,0,0,0)] if not seasonal else list(product([0,1], [0,1], [0,1], [12]))

    best_aic = np.inf
    best_result = None
    best_spec = None

    for order in orders:
        for sea in seas:
            try:
                model = SARIMAX(
                    train_y,
                    exog=exog_train,
                    order=order,
                    seasonal_order=sea,
                    enforce_stationarity=False,
                    enforce_invertibility=False,
                )
                result = model.fit(disp=False)
                if result.aic < best_aic:
                    best_aic = result.aic
                    best_result = result
                    best_spec = (order, sea)
            except Exception:
                continue
    return best_result, best_spec, best_aic


def fit_forecast_arima(train_y, horizon):
    result, spec, _ = search_sarimax(train_y, exog_train=None, seasonal=False, fast_mode=FAST_MODE)
    if result is None:
        raise ValueError("ARIMA fitting failed")
    fc = result.get_forecast(steps=horizon)
    return np.maximum(0, np.array(fc.predicted_mean)), np.maximum(0.0, np.asarray(fc.conf_int())), spec


def fit_forecast_sarima(train_y, horizon):
    result, spec, _ = search_sarimax(train_y, exog_train=None, seasonal=True, fast_mode=FAST_MODE)
    if result is None:
        raise ValueError("SARIMA fitting failed")
    fc = result.get_forecast(steps=horizon)
    return np.maximum(0, np.array(fc.predicted_mean)), np.maximum(0.0, np.asarray(fc.conf_int())), spec


def fit_forecast_sarimax(train_y, exog_train, exog_test):
    result, spec, _ = search_sarimax(train_y, exog_train=exog_train, seasonal=True, fast_mode=FAST_MODE)
    if result is None:
        raise ValueError("SARIMAX fitting failed")
    fc = result.get_forecast(steps=len(exog_test), exog=exog_test)
    return np.maximum(0, np.array(fc.predicted_mean)), np.maximum(0.0, np.asarray(fc.conf_int())), spec




def create_sequences_univariate(series, look_back=3):
    X, y = [], []
    for i in range(len(series) - look_back):
        X.append(series[i:i + look_back])
        y.append(series[i + look_back])
    X = np.array(X)
    y = np.array(y)
    return X.reshape((X.shape[0], X.shape[1], 1)), y


def create_sequences_multivariate(features, target, look_back=3):
    X, y = [], []
    for i in range(len(features) - look_back):
        X.append(features[i:i + look_back])
        y.append(target[i + look_back])
    return np.array(X), np.array(y)


# ================================================================
# PER-DISTRICT RANDOM SEEDING
# ================================================================
def seed_district(district_id):
    """Reseed before a district's neural models; see USE_KERAS_RANDOM_SEED in Section A."""
    import tensorflow as tf
    seed = SEED + int(district_id)
    if USE_KERAS_RANDOM_SEED:
        tf.keras.utils.set_random_seed(seed)   # v5: repeatable under Keras 3
    else:
        tf.random.set_seed(seed)               # v4: not repeatable under Keras 3


# ================================================================
# MC-DROPOUT LAYER
# ================================================================
# The layer class subclasses a Keras class, so it is built on first use
# inside a worker rather than at definition time in this kernel (cell 1).
_MC_DROPOUT_STATE = {"active": False}
_MC_DROPOUT_CLASS = None


def _mc_dropout_class():
    global _MC_DROPOUT_CLASS
    if _MC_DROPOUT_CLASS is None:
        import tensorflow as tf

        class _MCDropoutLayer(tf.keras.layers.Dropout):
            """Dropout that can also stay active at inference, for MC-dropout intervals.

            Identical to Dropout unless _MC_DROPOUT_STATE["active"] is True; then it
            drops units even when the model is called with training=False. Calling
            the model with training=True instead would also switch on GaussianNoise
            and make BatchNormalization use the statistics of the current batch,
            which would distort the interval. Only mc_dropout_interval() turns it on.
            """

            def call(self, inputs, training=None):
                return super().call(inputs, training=bool(training) or _MC_DROPOUT_STATE["active"])

        _MC_DROPOUT_CLASS = _MCDropoutLayer
    return _MC_DROPOUT_CLASS


def MCDropout(rate, **kwargs):
    return _mc_dropout_class()(rate, **kwargs)


# ================================================================
# REGULARIZED LSTM TRAINING WITH TIME-ORDERED VALIDATION
# ================================================================
LSTM_EPOCHS = 150
LSTM_BATCH_SIZE = 16
LSTM_PATIENCE = 15
LSTM_VAL_RATIO = 0.20

LSTM_SEARCH_SPACE = [
    # "huber" = tf.keras.losses.Huber() (delta 1.0), resolved inside the
    # builder so no Keras object is created in this kernel at definition time.
    {"units1": 64,  "units2": 32, "dropout": 0.20, "lr": 0.001,  "loss": "huber"},
    {"units1": 96,  "units2": 48, "dropout": 0.25, "lr": 0.001,  "loss": "huber"},
    {"units1": 128, "units2": 64, "dropout": 0.30, "lr": 0.0005, "loss": "huber"},
    {"units1": 64,  "units2": 32, "dropout": 0.25, "lr": 0.0005, "loss": "mae"},
]

# FAST_MODE now covers the neural search too, not only the ARIMA/SARIMA/
# SARIMAX grids. Under FAST_MODE the LSTM uses the single locked
# specification LSTM_SEARCH_SPACE[0] -- 64/32 units, dropout 0.20,
# lr 0.001, Huber loss -- which is the same specification train_compact_gru
# already hard-codes for the GRU, so all three neural models are then
# treated consistently: one regularized specification each, with early
# stopping, rather than a grid for LSTM and a fixed spec for GRU.
#
# Cost: this drops per-fold neural training from 9 fits to 3 (Univariate
# LSTM, Multivariate LSTM, GRU), i.e. about two thirds less neural work.
#
# This CHANGES RESULTS: candidates 2-4 no longer compete, so a district
# whose best LSTM would have been one of them now gets the locked one
# instead. That is why PIPELINE_CODE_VERSION is bumped and FAST_MODE is
# folded into the run signature -- old checkpoints must not be reused.
# Set FAST_MODE = False to restore the full four-candidate search.
LSTM_ACTIVE_SEARCH_SPACE = LSTM_SEARCH_SPACE[:1] if FAST_MODE else LSTM_SEARCH_SPACE
print(
    f"LSTM candidates per fit: {len(LSTM_ACTIVE_SEARCH_SPACE)}"
    f" (FAST_MODE={FAST_MODE})"
)


def time_based_train_val_split(X, y, val_ratio=0.20):
    n = len(X)
    val_size = max(1, int(n * val_ratio))
    split_idx = n - val_size

    X_train = X[:split_idx]
    y_train = y[:split_idx]
    X_val = X[split_idx:]
    y_val = y[split_idx:]

    return X_train, X_val, y_train, y_val


def build_regularized_lstm(
    input_shape,
    units1=96,
    units2=48,
    dropout=0.25,
    lr=0.001,
    loss="huber"
):
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras import regularizers
    from tensorflow.keras.layers import LSTM, Dense, BatchNormalization, GaussianNoise
    from tensorflow.keras.optimizers import Adam
    if loss == "huber":
        loss = tf.keras.losses.Huber()

    model = Sequential()

    model.add(GaussianNoise(0.01, input_shape=input_shape))

    model.add(
        LSTM(
            units1,
            return_sequences=True,
            kernel_regularizer=regularizers.l2(1e-4),
            recurrent_regularizer=regularizers.l2(1e-4)
        )
    )
    model.add(BatchNormalization())
    model.add(MCDropout(dropout))

    model.add(
        LSTM(
            units2,
            return_sequences=False,
            kernel_regularizer=regularizers.l2(1e-4),
            recurrent_regularizer=regularizers.l2(1e-4)
        )
    )
    model.add(BatchNormalization())
    model.add(MCDropout(dropout))

    model.add(
        Dense(
            32,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4)
        )
    )
    model.add(MCDropout(dropout / 2))
    model.add(Dense(1))

    model.compile(
        optimizer=Adam(
            learning_rate=lr,
            clipnorm=1.0
        ),
        loss=loss,
        metrics=["mae"]
    )

    return model


def train_tuned_lstm(X, y, model_label):
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    if len(X) < 30:
        return None, None, None

    X_tr, X_val, y_tr, y_val = time_based_train_val_split(
        X,
        y,
        val_ratio=LSTM_VAL_RATIO
    )

    best_model = None
    best_history = None
    best_params = None
    best_val_loss = np.inf

    for params in LSTM_ACTIVE_SEARCH_SPACE:
        model = build_regularized_lstm(
            input_shape=(X.shape[1], X.shape[2]),
            units1=params["units1"],
            units2=params["units2"],
            dropout=params["dropout"],
            lr=params["lr"],
            loss=params["loss"]
        )

        es = EarlyStopping(
            monitor="val_loss",
            patience=LSTM_PATIENCE,
            min_delta=1e-4,
            restore_best_weights=True
        )

        reduce_lr = ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-5,
            verbose=0
        )

        history = model.fit(
            X_tr,
            y_tr,
            validation_data=(X_val, y_val),
            epochs=LSTM_EPOCHS,
            batch_size=LSTM_BATCH_SIZE,
            shuffle=False,
            verbose=0,
            callbacks=[es, reduce_lr]
        )

        current_val_loss = np.nanmin(history.history.get("val_loss", [np.inf]))

        # Treat a non-finite validation loss as the worst possible score
        # rather than letting it poison the comparison. Two failure modes
        # this closes:
        #   - NaN < best_val_loss is False, so a candidate whose training
        #     diverged used to leave best_model as None. Harmless when four
        #     candidates competed, but under FAST_MODE there is only one, so
        #     a single NaN silently dropped this whole model family from the
        #     district's competition with no error raised.
        #   - Assigning NaN into best_val_loss also poisoned every later
        #     comparison, because finite < NaN is False too, so a good
        #     candidate after a diverged one could never win.
        # 'best_model is None' keeps the first candidate provisionally, which
        # is what 'anything < inf' already did, so runs with no divergence
        # select exactly the same model as before.
        if not np.isfinite(current_val_loss):
            current_val_loss = np.inf

        if best_model is None or current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            best_model = model
            best_history = history
            best_params = params.copy()

    print(
        f"{model_label} best params:",
        best_params,
        "| best val_loss:",
        round(best_val_loss, 5)
    )

    return best_model, best_history, best_params


# ================================================================
# RECURSIVE FORECASTING AND MC-DROPOUT INTERVALS
# ================================================================
# The model is called directly (model(x, training=False)) instead of
# model.predict(): repeated .predict() calls inside a loop leak memory in
# TF/Keras. The result equals .predict() up to float32 rounding.
#
# The forecast_paths_* functions run n_paths forecast paths as one batch.
# With MCDropout.mc_active off every path is identical, so path 0 is the
# point forecast. With it on, each path gets its own dropout masks, which
# gives the MC-dropout paths with one model call per step instead of one
# call per path per step.
TARGET_HISTORY_FEATURES = [
    "malaria_cases_lag1", "malaria_cases_lag2", "malaria_cases_lag3",
    "malaria_cases_rollmean_3", "malaria_cases_rollmean_6",
]


def forecast_paths_univariate(model, history_scaled, horizon, scaler, n_paths=1):
    history = np.repeat(
        np.asarray(history_scaled, dtype=float).reshape(1, -1)[:, -LOOK_BACK:], n_paths, axis=0
    )
    preds = np.zeros((n_paths, horizon))
    for step in range(horizon):
        x = history[:, -LOOK_BACK:].reshape(n_paths, LOOK_BACK, 1).astype(np.float32)
        pred_scaled = model(x, training=False).numpy().reshape(-1)
        history = np.concatenate([history, pred_scaled[:, None]], axis=1)
        preds[:, step] = np.maximum(
            0.0, scaler.inverse_transform(pred_scaled.reshape(-1, 1)).reshape(-1)
        )
    return preds


def recursive_forecast_univariate(model, history_scaled, horizon, scaler):
    return forecast_paths_univariate(model, history_scaled, horizon, scaler)[0]


def forecast_paths_multivariate(model, feature_seq, future_features_raw, feature_scaler,
                                target_scaler, target_history, n_paths=1):
    """Recursive multivariate forecast with target-derived predictors rebuilt each step.

    feature_seq         : scaled training matrix, column 0 = scaled target.
    future_features_raw : UNSCALED feature rows of the forecast months, columns
                          in the order feature_scaler was fitted on. Climate
                          values and climate lags are used as given.
    target_history      : observed malaria_cases up to the forecast origin.

    A sequence row for month t holds target(t) and features(t); the model
    predicts month t from the previous LOOK_BACK rows. The malaria lag and
    rolling-mean features of month t use only months <= t-1: observed values
    before the forecast origin, the model's own predictions after it. The
    observed values of the forecast period are never used, so validation
    folds, the 2025 test and the 2026 forecast follow the same procedure.
    Returns an (n_paths, horizon) array.
    """
    cols = list(future_features_raw.columns)
    future_raw = future_features_raw.to_numpy(dtype=float)
    lag_positions = [(name, cols.index(name)) for name in TARGET_HISTORY_FEATURES if name in cols]

    seq = np.asarray(feature_seq, dtype=float)[-LOOK_BACK:]
    history = np.repeat(seq[None, :, :], n_paths, axis=0)
    past = np.asarray(target_history, dtype=float).reshape(-1)[-6:]
    if len(past) < 6:
        pad_value = past[0] if len(past) else 0.0
        past = np.concatenate([np.repeat(pad_value, 6 - len(past)), past])
    target_hist = np.repeat(past[None, :], n_paths, axis=0)

    horizon = len(future_raw)
    preds = np.zeros((n_paths, horizon))
    for step in range(horizon):
        pred_scaled = model(history.astype(np.float32), training=False).numpy().reshape(-1)
        pred = np.maximum(
            0.0, target_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).reshape(-1)
        )
        preds[:, step] = pred

        derived = {
            "malaria_cases_lag1": target_hist[:, -1],
            "malaria_cases_lag2": target_hist[:, -2],
            "malaria_cases_lag3": target_hist[:, -3],
            "malaria_cases_rollmean_3": target_hist[:, -3:].mean(axis=1),
            "malaria_cases_rollmean_6": target_hist[:, -6:].mean(axis=1),
        }
        rows = np.repeat(future_raw[step][None, :], n_paths, axis=0)
        for name, position in lag_positions:
            rows[:, position] = derived[name]
        scaled = feature_scaler.transform(pd.DataFrame(rows, columns=cols))
        next_rows = np.concatenate([pred_scaled[:, None], scaled], axis=1)
        history = np.concatenate([history[:, 1:, :], next_rows[:, None, :]], axis=1)
        target_hist = np.concatenate([target_hist[:, 1:], pred[:, None]], axis=1)
    return preds


def recursive_forecast_multivariate(model, feature_seq, future_features_raw, feature_scaler,
                                    target_scaler, target_history):
    return forecast_paths_multivariate(
        model, feature_seq, future_features_raw, feature_scaler, target_scaler, target_history
    )[0]


def mc_dropout_interval(forecast_paths_fn, n_iter=None):
    """95% MC-dropout interval: 2.5th and 97.5th percentiles over n_iter paths.

    forecast_paths_fn(n_paths) must return an (n_paths, horizon) array. Only
    the MCDropout layers become stochastic; the point forecast is computed
    separately with MC dropout off.
    """
    n_iter = MC_DROPOUT_ITER if n_iter is None else n_iter
    _MC_DROPOUT_STATE["active"] = True
    try:
        paths = forecast_paths_fn(n_iter)
    finally:
        _MC_DROPOUT_STATE["active"] = False
    return np.percentile(paths, 2.5, axis=0), np.percentile(paths, 97.5, axis=0)


# ================================================================
# RESOURCE-CONSCIOUS GRU HELPERS
# ================================================================
def build_compact_gru(input_shape, units1=64, units2=32, dropout=0.20, lr=0.001):
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras import regularizers
    from tensorflow.keras.layers import GRU, Dense, BatchNormalization, GaussianNoise
    from tensorflow.keras.optimizers import Adam
    model = Sequential([
        GaussianNoise(0.01, input_shape=input_shape),
        GRU(units1, return_sequences=True,
            kernel_regularizer=regularizers.l2(1e-4),
            recurrent_regularizer=regularizers.l2(1e-4)),
        BatchNormalization(),
        MCDropout(dropout),
        GRU(units2, return_sequences=False,
            kernel_regularizer=regularizers.l2(1e-4),
            recurrent_regularizer=regularizers.l2(1e-4)),
        BatchNormalization(),
        MCDropout(dropout),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(
        optimizer=Adam(learning_rate=lr, clipnorm=1.0),
        loss=tf.keras.losses.Huber(),
        metrics=["mae"],
    )
    return model


def train_compact_gru(X, y, model_label="GRU"):
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    if len(X) < 30:
        return None, None, None
    X_tr, X_val, y_tr, y_val = time_based_train_val_split(X, y, val_ratio=0.20)
    params = {"units1": 64, "units2": 32, "dropout": 0.20, "lr": 0.001}
    model = build_compact_gru((X.shape[1], X.shape[2]), **params)
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=GRU_PATIENCE, min_delta=1e-4, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5, verbose=0),
    ]
    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=GRU_EPOCHS,
        batch_size=GRU_BATCH_SIZE,
        shuffle=False,
        verbose=0,
        callbacks=callbacks,
    )
    print(model_label, "parameters:", params, "| epochs:", len(history.history.get("loss", [])))
    return model, history, params


# ================================================================
# XGBOOST TIME-SERIES HELPERS
# Outcome lags are built inside each training fold and updated recursively,
# avoiding use of future observed malaria counts during forecasting.
# ================================================================
def make_xgb_supervised(y, exog=None, look_back=3):
    y = np.asarray(y, dtype=float).reshape(-1)
    if exog is None:
        exog = np.empty((len(y), 0), dtype=float)
    exog = np.asarray(exog, dtype=float)
    rows, targets = [], []
    for t in range(look_back, len(y)):
        lag_values = [y[t - lag] for lag in range(1, look_back + 1)]
        rows.append(lag_values + exog[t].tolist())
        targets.append(y[t])
    return np.asarray(rows, dtype=float), np.asarray(targets, dtype=float)


def recursive_forecast_xgboost(model, history_y, future_exog=None, horizon=12, look_back=3):
    history = list(np.asarray(history_y, dtype=float).reshape(-1))
    if future_exog is None:
        future_exog = np.empty((horizon, 0), dtype=float)
    future_exog = np.asarray(future_exog, dtype=float)
    preds = []
    for step in range(horizon):
        lag_values = [history[-lag] for lag in range(1, look_back + 1)]
        row = np.asarray(lag_values + future_exog[step].tolist(), dtype=float).reshape(1, -1)
        pred = max(0.0, float(model.predict(row)[0]))
        preds.append(pred)
        history.append(pred)
    return np.asarray(preds)


def fit_forecast_xgboost(train_y, train_exog, future_exog, horizon, look_back=3):
    X_fit, y_fit = make_xgb_supervised(train_y, train_exog, look_back=look_back)
    if len(X_fit) < 24:
        raise ValueError("Insufficient supervised observations for XGBoost.")
    model = XGBRegressor(**XGB_PARAMS)
    model.fit(X_fit, y_fit)
    fitted = np.maximum(0.0, model.predict(X_fit))
    residual_sd = float(np.std(y_fit - fitted, ddof=1)) if len(y_fit) > 1 else np.nan
    pred = recursive_forecast_xgboost(
        model, train_y, future_exog=future_exog, horizon=horizon, look_back=look_back
    )
    return pred, model, residual_sd


mark_section_checkpoint("B_helper_functions")

## SECTION C. READ RAW DATA

In [ ]:
# ================================================================
# SECTION C. READ RAW DATA

In [ ]:
# ================================================================
if not os.path.exists(MALARIA_FILE):
    raise FileNotFoundError(f"Malaria file not found: {MALARIA_FILE}")
if not os.path.exists(CLIMATE_FILE):
    raise FileNotFoundError(f"Climate file not found: {CLIMATE_FILE}")

malaria_raw = pd.read_csv(MALARIA_FILE)
climate_raw = pd.read_csv(CLIMATE_FILE)

print("Malaria raw shape:", malaria_raw.shape)
print("Climate raw shape:", climate_raw.shape)
display(malaria_raw.head())
display(climate_raw.head())
mark_section_checkpoint("C_read_raw_data")

## SECTION D. INSPECT COLUMN STRUCTURE

In [ ]:
# ================================================================
# SECTION D. INSPECT COLUMN STRUCTURE

In [ ]:
# ================================================================
print("Malaria columns:", malaria_raw.columns.tolist())
print("Climate columns:", climate_raw.columns.tolist())

# One row per province per month, from the DDC malaria MIS overview.
# The case count is All_All: Thai + M1 + M2 migrants, all species.
malaria_year_col = first_existing(malaria_raw, ["year_ce", "year"])
malaria_month_col = first_existing(malaria_raw, ["month", "Month"])
malaria_district_id_col = first_existing(malaria_raw, ["province_id"])
malaria_case_col = first_existing(malaria_raw, ["All_All", "malaria_cases"])
# The malaria file has Thai province names only; English names come from
# the climate file. Aggregated counts carry no sex or age.
malaria_district_name_col = None
sex_col = None
age_col = None
province_col_mal = None

climate_district_id_col = first_existing(climate_raw, ["ADM2_CODE", "district_id", "Patient_Site_District_ID"])
climate_district_col = first_existing(climate_raw, ["district", "District", "Patient_Site_District_Name"])
climate_province_col = first_existing(climate_raw, ["province", "Province", "Patient_Site_Province_Name"], required=False)
climate_year_col = first_existing(climate_raw, ["year", "Year"])
climate_month_col = first_existing(climate_raw, ["month", "Month"])
rain_col = first_existing(climate_raw, ["rain_mm_sum", "rainfall", "precipitation", "Rainfall"])
temp_col = first_existing(climate_raw, ["temp_c_mean", "temperature", "Temperature", "mean_temperature"])
rh_col = first_existing(climate_raw, ["RH_mean", "relative_humidity", "humidity", "Relative_Humidity"])
intervention_col_climate = first_existing(climate_raw, ["intervention", "Intervention", "intervention_index", "control_intervention"], required=False)
mark_section_checkpoint("D_inspect_columns")

## SECTION E. DATA INTEGRATION PART 1: PREPARE MONTHLY PROVINCE-LEVEL MALARIA
Explanation:
The public input is already one row per province-month, so this section checks it instead of aggregating case records.

In [ ]:
# ================================================================
# SECTION E. DATA INTEGRATION PART 1: PREPARE MONTHLY PROVINCE-LEVEL MALARIA
# Explanation:
# The public input is already one row per province-month, so this section
# checks it instead of aggregating case records.


In [ ]:
# ================================================================
malaria = malaria_raw[[malaria_district_id_col, malaria_year_col, malaria_month_col, malaria_case_col]].copy()
malaria.columns = [malaria_district_id_col, "year", "month", "malaria_cases"]
malaria = malaria.apply(pd.to_numeric, errors="coerce")
if malaria.isna().any().any():
    raise ValueError("Malaria file has missing or non-numeric province, year, month or case values.")
malaria = malaria.astype({malaria_district_id_col: int, "year": int, "month": int, "malaria_cases": float})
malaria = malaria[(malaria["year"] >= HIST_START_YEAR) & (malaria["year"] <= HIST_END_YEAR)].copy()

agg_cols = [malaria_district_id_col, "year", "month"]
if malaria.duplicated(agg_cols).any():
    raise ValueError("Malaria file must have one row per province-month.")

# Section G reads a missing province-month as a true zero, so a month that
# was never downloaded would silently become zero cases. Refuse that.
downloaded = set(zip(malaria["year"], malaria["month"]))
missing_months = [
    f"{y}-{m:02d}"
    for y in range(HIST_START_YEAR, HIST_END_YEAR + 1)
    for m in range(1, 13)
    if (y, m) not in downloaded
]
if missing_months:
    raise ValueError(f"Malaria file lacks {len(missing_months)} months: {missing_months[:6]} ...")
if malaria.groupby(["year", "month"])[malaria_district_id_col].nunique().nunique() != 1:
    raise ValueError("Every month must list the same number of provinces.")

malaria_monthly = malaria.copy()
print("Monthly malaria shape:", malaria_monthly.shape)
print("Cases by year:", malaria_monthly.groupby("year")["malaria_cases"].sum().astype(int).to_dict())
display(malaria_monthly.head())
mark_section_checkpoint("E_malaria_monthly")


## SECTION F. DATA INTEGRATION PART 2: PREPARE CLIMATE DATA
Explanation:

In [ ]:
# ================================================================
# SECTION F. DATA INTEGRATION PART 2: PREPARE CLIMATE DATA
# Explanation:
# The climate file is monthly district-level; it is averaged to province below.
# We standardize key columns and keep rainfall, temperature, RH, and optional intervention.

In [ ]:
# ================================================================
climate = climate_raw.copy()
rename_map = {
    climate_district_id_col: "ADM2_CODE",
    climate_district_col: "district",
    climate_year_col: "year",
    climate_month_col: "month",
    rain_col: "rain_mm_sum",
    temp_col: "temp_c_mean",
    rh_col: "RH_mean",
}
if climate_province_col:
    rename_map[climate_province_col] = "province"
if intervention_col_climate:
    rename_map[intervention_col_climate] = "intervention"
climate = climate.rename(columns=rename_map)

keep_cols = ["ADM2_CODE", "district", "year", "month", "rain_mm_sum", "temp_c_mean", "RH_mean"]
if "province" in climate.columns:
    keep_cols.append("province")
if "intervention" in climate.columns:
    keep_cols.append("intervention")
climate = climate[keep_cols].copy()

for c in ["ADM2_CODE", "year", "month", "rain_mm_sum", "temp_c_mean", "RH_mean"]:
    climate[c] = pd.to_numeric(climate[c], errors="coerce")
if "intervention" in climate.columns:
    if climate["intervention"].dtype == "object":
        climate = pd.get_dummies(climate, columns=["intervention"], prefix="intervention", drop_first=False)
    else:
        climate["intervention"] = pd.to_numeric(climate["intervention"], errors="coerce")

climate = climate.dropna(subset=["ADM2_CODE", "year", "month"]).copy()
climate[["ADM2_CODE", "year", "month"]] = climate[["ADM2_CODE", "year", "month"]].astype(int)
climate = climate[(climate["year"] >= HIST_START_YEAR) & (climate["year"] <= HIST_END_YEAR)].copy()
if "province" not in climate.columns:
    climate["province"] = "Unknown"

# The malaria counts are per province, so average the district climate
# values within each province (unweighted mean over districts). From here
# on ADM2_CODE holds the 2-digit province code (district code // 100) and
# "district" holds the province name, so later sections run unchanged.
climate["ADM2_CODE"] = climate["ADM2_CODE"] // 100
province_names = climate.groupby("ADM2_CODE")["province"].first()
climate = (
    climate.drop(columns=["district", "province"])
           .groupby(["ADM2_CODE", "year", "month"], as_index=False)
           .mean(numeric_only=True)
)
climate["province"] = climate["ADM2_CODE"].map(province_names)
climate["district"] = climate["province"]

unmatched = sorted(set(malaria_monthly[malaria_district_id_col]) ^ set(climate["ADM2_CODE"]))
if unmatched:
    raise ValueError(f"Province codes found in only one of the two files: {unmatched}")

print("Prepared climate shape:", climate.shape)
display(climate.head())
mark_section_checkpoint("F_climate_prepared")

## SECTION G. DATA INTEGRATION PART 3: CREATE COMPLETE DISTRICT-MONTH PANEL
Explanation:
This ensures that every district has all months from 2016-2025.

In [ ]:
# ================================================================
# SECTION G. DATA INTEGRATION PART 3: CREATE COMPLETE DISTRICT-MONTH PANEL
# Explanation:
# This ensures that every district has all months from 2016-2025.
# Missing malaria after merging is interpreted as a true zero-case month.

In [ ]:
# ================================================================
all_districts = climate[["ADM2_CODE", "district", "province"]].drop_duplicates().copy()
all_months = build_month_index(HIST_START_YEAR, HIST_END_YEAR)
panel = (
    all_districts.assign(key=1)
                 .merge(all_months.assign(key=1), on="key")
                 .drop(columns="key")
)

malaria_monthly = malaria_monthly.rename(columns={malaria_district_id_col: "ADM2_CODE"})
if malaria_district_name_col:
    malaria_monthly = malaria_monthly.rename(columns={malaria_district_name_col: "district_name_malaria"})
if province_col_mal:
    malaria_monthly = malaria_monthly.rename(columns={province_col_mal: "province_name_malaria"})

panel_malaria = panel.merge(malaria_monthly, on=["ADM2_CODE", "year", "month"], how="left")
panel_malaria["malaria_cases"] = panel_malaria["malaria_cases"].fillna(0).astype(float)

if "district_name_malaria" in panel_malaria.columns:
    panel_malaria["district_final"] = panel_malaria["district_name_malaria"].fillna(panel_malaria["district"])
else:
    panel_malaria["district_final"] = panel_malaria["district"]

panel_data = panel_malaria.merge(climate, on=["ADM2_CODE", "district", "province", "year", "month"], how="left")
panel_data = add_date_columns(panel_data)

print("Integrated panel shape:", panel_data.shape)
display(panel_data.head())
mark_section_checkpoint("G_integrated_panel")

## SECTION H. MERGE QUALITY CHECK

In [ ]:
# ================================================================
# SECTION H. MERGE QUALITY CHECK

In [ ]:
# ================================================================
print("Missing malaria cases:", panel_data["malaria_cases"].isna().sum())
print("Missing rainfall:", panel_data["rain_mm_sum"].isna().sum())
print("Missing temperature:", panel_data["temp_c_mean"].isna().sum())
print("Missing RH:", panel_data["RH_mean"].isna().sum())
print("District count in final panel:", panel_data["ADM2_CODE"].nunique())
mark_section_checkpoint("H_merge_quality_check")

## SECTION I. DATA PREPARATION: KNN PARAMETER VALIDATION AND MISSING-VALUE IMPUTATION

True zero malaria counts are retained. KNN is applied only to predictor variables after district-wise temporal interpolation. Candidate values `k = 3, 5, 7, 9, 11` are evaluated through artificial masking of complete training-period predictor records. The validation sample is capped to avoid excessive memory use and runtime.


In [ ]:
# ================================================================
# SECTION I. KNN IMPUTATION PARAMETER VALIDATION
# ================================================================


In [ ]:
model_df = panel_data.copy()
model_df = model_df.sort_values(["ADM2_CODE", "date"]).reset_index(drop=True)

numeric_predictors = ["rain_mm_sum", "temp_c_mean", "RH_mean"]
intervention_dummy_cols = [c for c in model_df.columns if c.startswith("intervention")]
numeric_predictors += intervention_dummy_cols

for col in ["rain_mm_sum", "temp_c_mean", "RH_mean"]:
    if col in model_df.columns:
        model_df[col] = model_df.groupby("ADM2_CODE")[col].transform(
            lambda x: x.interpolate(method="linear", limit_direction="both")
        )

imputer_cols = [c for c in numeric_predictors if c in model_df.columns]
if not imputer_cols:
    raise ValueError("No numeric predictor columns were available for KNN validation.")


def validate_knn_neighbors(
    df,
    columns,
    candidates=(3, 5, 7, 9, 11),
    max_rows=2000,
    mask_fraction=0.10,
    repeats=1,
    random_state=42,
):
    """Select k using standardized artificial-masking error on training data only."""
    complete = (
        df.loc[df["year"] <= TRAIN_END_YEAR, columns]
        .apply(pd.to_numeric, errors="coerce")
        .dropna()
        .astype(float)
    )
    if len(complete) < max(candidates) + 2:
        raise ValueError("Insufficient complete training records for KNN parameter validation.")
    if len(complete) > max_rows:
        complete = complete.sample(n=max_rows, random_state=random_state)

    scaler = StandardScaler()
    complete_z = scaler.fit_transform(complete)
    rng = np.random.default_rng(random_state)
    rows = []

    for repeat in range(repeats):
        masked = complete_z.copy()
        mask = rng.random(masked.shape) < mask_fraction
        # Retain at least one observed predictor in every row.
        fully_masked = np.where(mask.all(axis=1))[0]
        for row_idx in fully_masked:
            mask[row_idx, rng.integers(0, mask.shape[1])] = False
        if not mask.any():
            mask[rng.integers(0, mask.shape[0]), rng.integers(0, mask.shape[1])] = True
        masked[mask] = np.nan

        for k in candidates:
            imputed = KNNImputer(n_neighbors=k, weights="uniform").fit_transform(masked)
            error = complete_z[mask] - imputed[mask]
            rows.append({
                "repeat": repeat + 1,
                "k": int(k),
                "n_validation_rows": int(len(complete)),
                "n_masked_cells": int(mask.sum()),
                "standardized_MAE": float(np.mean(np.abs(error))),
                "standardized_RMSE": float(np.sqrt(np.mean(error ** 2))),
            })

    detail = pd.DataFrame(rows)
    summary = (
        detail.groupby("k", as_index=False)
        .agg(
            standardized_MAE=("standardized_MAE", "mean"),
            standardized_RMSE=("standardized_RMSE", "mean"),
            n_validation_rows=("n_validation_rows", "max"),
            n_masked_cells=("n_masked_cells", "sum"),
        )
        .sort_values(["standardized_RMSE", "standardized_MAE", "k"])
        .reset_index(drop=True)
    )
    selected_k = int(summary.iloc[0]["k"])
    return selected_k, summary, detail


try:
    SELECTED_KNN_K, knn_validation_summary, knn_validation_detail = validate_knn_neighbors(
        model_df,
        imputer_cols,
        candidates=KNN_CANDIDATES,
        max_rows=KNN_VALIDATION_MAX_ROWS,
        mask_fraction=KNN_MASK_FRACTION,
        repeats=KNN_VALIDATION_REPEATS,
        random_state=SEED,
    )
except ValueError as exc:
    # A prespecified fallback prevents workflow failure in very small pilot subsets.
    SELECTED_KNN_K = 5
    knn_validation_summary = pd.DataFrame({
        "k": KNN_CANDIDATES,
        "standardized_MAE": np.nan,
        "standardized_RMSE": np.nan,
        "n_validation_rows": np.nan,
        "n_masked_cells": np.nan,
    })
    knn_validation_detail = pd.DataFrame()
    print(f"KNN validation fallback used: {exc}")

print("Selected KNN k:", SELECTED_KNN_K)
display(knn_validation_summary)

# Fit the scaling and imputation objects on the training period only. This
# prevents the 2025 external-test predictors from becoming KNN donor records.
knn_train_mask = model_df["year"] <= TRAIN_END_YEAR
knn_scaler = StandardScaler()
knn_train_z = knn_scaler.fit_transform(model_df.loc[knn_train_mask, imputer_cols])
knn_all_z = knn_scaler.transform(model_df[imputer_cols])
knn_imputer = KNNImputer(n_neighbors=SELECTED_KNN_K, weights="uniform")
knn_imputer.fit(knn_train_z)
model_df[imputer_cols] = knn_scaler.inverse_transform(knn_imputer.transform(knn_all_z))

knn_validation_summary.to_csv(f"{OUTPUT_DIR}/knn_parameter_validation_summary.csv", index=False)
knn_validation_detail.to_csv(f"{OUTPUT_DIR}/knn_parameter_validation_detail.csv", index=False)
pd.DataFrame({"selected_k": [SELECTED_KNN_K]}).to_csv(
    f"{OUTPUT_DIR}/knn_selected_parameter.csv", index=False
)

print(model_df[imputer_cols].isnull().sum())
mark_section_checkpoint("I_knn_imputation", selected_k=SELECTED_KNN_K)

## SECTION J. CHARACTERISTICS TABLES (OUTPUT 1 / OBJECTIVE 1)
Explanation:

In [ ]:
# ================================================================
# SECTION J. CHARACTERISTICS TABLES (OUTPUT 1 / OBJECTIVE 1)
# Explanation:
# These descriptive tables summarize person, time, and place.

In [ ]:
# ================================================================
malaria_desc = malaria.copy()

if sex_col:
    sex_table = malaria_desc[sex_col].value_counts(dropna=False).reset_index()
    sex_table.columns = ["Sex", "Count"]
else:
    sex_table = pd.DataFrame({"Sex": ["Not available"], "Count": [np.nan]})

if age_col:
    malaria_desc[age_col] = pd.to_numeric(malaria_desc[age_col], errors="coerce")
    age_table = malaria_desc[age_col].describe().to_frame(name="Age").reset_index()
else:
    age_table = pd.DataFrame({"index": ["Not available"], "Age": [np.nan]})

year_table = malaria_desc.groupby("year", as_index=False)["malaria_cases"].sum().rename(columns={"malaria_cases": "Case_Count"})

if province_col_mal:
    province_table = malaria_desc.groupby(province_col_mal, as_index=False).size().rename(columns={province_col_mal: "Province", "size": "Case_Count"}).sort_values("Case_Count", ascending=False)
else:
    province_table = model_df.groupby("province", as_index=False)["malaria_cases"].sum().rename(columns={"malaria_cases": "Case_Count", "province": "Province"}).sort_values("Case_Count", ascending=False)

print("Characteristics tables prepared")
display(sex_table.head())
display(year_table.head())
display(province_table.head())
mark_section_checkpoint("J_characteristics_tables")

## SECTION K. ENCODING AND FEATURE ENGINEERING
Explanation:
Matching the workflow figure:
- Encode any categorical intervention variables numerically
- Create lag features for malaria and climate (1, 2, 3 months)
- Create seasonal variables and moving averages

In [ ]:
# ================================================================
# SECTION K. ENCODING AND FEATURE ENGINEERING
# Explanation:
# Matching the workflow figure:
# - Encode any categorical intervention variables numerically
# - Create lag features for malaria and climate (1, 2, 3 months)
# - Create seasonal variables and moving averages

In [ ]:
# ================================================================
model_df = add_cyclical_month_features(model_df)
model_df = add_rolling_features(model_df, "ADM2_CODE", "malaria_cases", windows=(3, 6))
model_df = create_lag_features(model_df, "ADM2_CODE", ["malaria_cases", "rain_mm_sum", "temp_c_mean", "RH_mean"], lags=(1, 2, 3))

# Optional intervention lags if available
for c in intervention_dummy_cols:
    model_df = create_lag_features(model_df, "ADM2_CODE", [c], lags=(1, 2, 3))

# Remove the first 3 months of each district because lag3 is unavailable
model_df["district_month_index"] = model_df.groupby("ADM2_CODE").cumcount() + 1
model_df = model_df[model_df["district_month_index"] > 3].copy()

print("Feature-engineered dataset shape:", model_df.shape)
display(model_df.head())

Explanation:

In [ ]:
# ================================================================
# Explanation:
# The climate variables are retained according to the conceptual workflow and
# domain relevance, then passed forward into SARIMAX and feature engineering.
# ================================================================
selected_env_features = [c for c in ["rain_mm_sum", "temp_c_mean", "RH_mean"] + intervention_dummy_cols if c in model_df.columns]
selected_env_features = dedup_keep_order(selected_env_features)
selected_env_table = pd.DataFrame({"selected_environmental_predictor": selected_env_features})
display(selected_env_table)
mark_section_checkpoint("K_feature_engineering")

In [ ]:
# ================================================================

## SECTION M. OUTLIER CLUSTERING AND MANAGEMENT
Explanation:
- K-means clustering
- IQR-based outlier detection
- If deleting outliers would remove too much data, keep the row and cap the value instead

In [ ]:
# ================================================================
# SECTION M. OUTLIER CLUSTERING AND MANAGEMENT
# Explanation:
# The workflow requests:
# - K-means clustering
# - IQR-based outlier detection
# - If deleting outliers would remove too much data, keep the row and cap the value instead
# Here, outliers are handled district-by-district.

In [ ]:
# ================================================================
# K-MEANS CLUSTERING AND IQR OUTLIER MANAGEMENT
# ================================================================

cluster_features = [
    c for c in [
        "rain_mm_sum",
        "temp_c_mean",
        "RH_mean",
        "month_sin",
        "month_cos"
    ]
    if c in model_df.columns
]

# ------------------------------------------------------------
# Prepare clustering dataset
# ------------------------------------------------------------

cluster_work = (
    model_df[cluster_features]
    .copy()
    .ffill()
    .bfill()
)

cluster_work = cluster_work.fillna(
    cluster_work.median(numeric_only=True)
)

# ------------------------------------------------------------
# K-Means clustering
# ------------------------------------------------------------

kmeans = KMeans(
    n_clusters=4,
    random_state=SEED,
    n_init=10
)

model_df["kmeans_cluster"] = kmeans.fit_predict(
    cluster_work
)

# ------------------------------------------------------------
# IQR-based outlier detection and capping
# ------------------------------------------------------------

bounds = []
clipped_values = []
outlier_flags = []

for district_id, g in model_df.groupby("ADM2_CODE"):

    q1 = g["malaria_cases"].quantile(0.25)
    q3 = g["malaria_cases"].quantile(0.75)

    iqr = q3 - q1

    lower = max(0, q1 - 1.5 * iqr)
    upper = q3 + 1.5 * iqr

    is_outlier = (
        (g["malaria_cases"] < lower) |
        (g["malaria_cases"] > upper)
    )

    remain_ratio = (~is_outlier).mean()

    # --------------------------------------------------------
    # Outlier capping
    # --------------------------------------------------------

    capped = g["malaria_cases"].clip(
        lower=0,
        upper=upper
    )

    bounds.append(
        pd.DataFrame({
            "ADM2_CODE": [district_id],
            "lower_bound": [lower],
            "upper_bound": [upper],
            "remain_ratio_if_removed": [remain_ratio]
        })
    )

    clipped_values.append(capped)
    outlier_flags.append(is_outlier)

# ------------------------------------------------------------
# Combine outputs
# ------------------------------------------------------------

outlier_bounds_df = pd.concat(
    bounds,
    ignore_index=True
)

model_df["malaria_cases_outlier"] = (
    pd.concat(outlier_flags)
    .sort_index()
    .astype(int)
)

model_df["malaria_cases_capped"] = (
    pd.concat(clipped_values)
    .sort_index()
)

# Final target variable
model_df["target_cases"] = (
    model_df["malaria_cases_capped"]
)

# ------------------------------------------------------------
# Preview outputs
# ------------------------------------------------------------

display(outlier_bounds_df.head())

print(
    model_df[
        [
            "malaria_cases",
            "malaria_cases_capped",
            "malaria_cases_outlier"
        ]
    ].head()
)
mark_section_checkpoint("M_outlier_management")

## SECTION N. NORMALIZATION (MIN-MAX SCALE)
Explanation:

In [ ]:
# ================================================================
# SECTION N. NORMALIZATION (MIN-MAX SCALE)
# Explanation:
# Scaling is applied to predictors. The target itself is only scaled inside LSTM blocks.

In [ ]:
# ================================================================
base_feature_cols = [
    "month", "month_sin", "month_cos", "kmeans_cluster",
    "malaria_cases_lag1", "malaria_cases_lag2", "malaria_cases_lag3",
    "malaria_cases_rollmean_3", "malaria_cases_rollmean_6",
    "rain_mm_sum_lag1", "rain_mm_sum_lag2", "rain_mm_sum_lag3",
    "temp_c_mean_lag1", "temp_c_mean_lag2", "temp_c_mean_lag3",
    "RH_mean_lag1", "RH_mean_lag2", "RH_mean_lag3",
] + selected_env_features

for c in intervention_dummy_cols:
    for lag in (1, 2, 3):
        lag_name = f"{c}_lag{lag}"
        if lag_name in model_df.columns:
            base_feature_cols.append(lag_name)

feature_cols = [c for c in base_feature_cols if c in model_df.columns]
model_df[feature_cols] = model_df[feature_cols].replace([np.inf, -np.inf], np.nan)
for c in feature_cols:
    model_df[c] = model_df.groupby("ADM2_CODE")[c].transform(lambda x: x.fillna(x.median()))
    model_df[c] = model_df[c].fillna(model_df[c].median())
mark_section_checkpoint("N_normalization")

## SECTION O. TREND / SEASONAL VISUAL CHECK (optional output figure)

In [ ]:
# ================================================================
# SECTION O. TREND / SEASONAL VISUAL CHECK (optional output figure)

In [ ]:
# ================================================================
national_monthly = model_df.groupby("date", as_index=False)["target_cases"].sum()
plt.figure(figsize=(14, 4))
plt.plot(national_monthly["date"], national_monthly["target_cases"])
plt.title("Thailand monthly malaria burden (capped target) 2016-2025")
plt.xlabel("Month")
plt.ylabel("Cases")
plt.grid(True, alpha=0.3)
plt.show()
mark_section_checkpoint("O_trend_visual_check")

## SECTION P. TRAIN / TEST SPLIT
Explanation:
- Train: 2016-2024
- Test: 2025

In [ ]:
# ================================================================
# SECTION P. TRAIN / TEST SPLIT
# Explanation:
# - Train: 2016-2024
# - Test: 2025
# In addition, rolling-origin validation is used inside training districts.

In [ ]:
# ================================================================
train_data = model_df[model_df["year"] <= TRAIN_END_YEAR].copy()
test_data = model_df[model_df["year"] == TEST_YEAR].copy()

fold_plan = build_expanding_yearly_folds(
    train_data, start_year=HIST_START_YEAR,
    validation_years=EXPANDING_VALIDATION_YEARS,
)
if [fold["validation_year"] for fold in fold_plan] != EXPANDING_VALIDATION_YEARS:
    raise ValueError("The expected five validation years are not present in the training data.")
validation_plan = pd.DataFrame([
    {"stage": f"Fold {i}", "training_years": f"{HIST_START_YEAR}-{fold['train_end_year']}",
     "evaluation_year": fold["validation_year"], "horizon_months": TEST_HORIZON}
    for i, fold in enumerate(fold_plan, start=1)
] + [
    {"stage": "External test", "training_years": f"{HIST_START_YEAR}-{TRAIN_END_YEAR}",
     "evaluation_year": TEST_YEAR, "horizon_months": TEST_HORIZON},
    {"stage": "Final forecast", "training_years": f"{HIST_START_YEAR}-{HIST_END_YEAR}",
     "evaluation_year": FORECAST_YEAR, "horizon_months": TEST_HORIZON},
])
display(validation_plan)
validation_plan.to_csv(os.path.join(OUTPUT_DIR, "validation_plan.csv"), index=False)

print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)
mark_section_checkpoint("P_train_test_split")

## SECTION Q. SEVEN-MODEL DEVELOPMENT AND VALIDATION

The runner validates ARIMA, SARIMA, SARIMAX, Univariate LSTM, Multivariate LSTM, GRU, and XGBoost on 2020–2024. Each district's selected model must have a valid score on all five folds; 2025 is evaluated only after selection. Duplicate district-month records sharing an ADM2_CODE are reported and excluded because they cannot form an unambiguous 12-month series. The primary selection criterion is mean RMSE. The GRU and XGBoost specifications are intentionally compact to control runtime.


In [ ]:
# SECTION Q. SEVEN-MODEL RUNNER


In [ ]:
# ================================================================
# SECTION Q. SEVEN-MODEL RUNNER
# Models: ARIMA, SARIMA, SARIMAX, Univariate LSTM,
# Multivariate LSTM, GRU, and XGBoost.
# Selection: lowest mean validation RMSE, then MAE and sMAPE.
# ================================================================
# Memory-safe revision: per-district results are checkpointed to
# DISTRICT_REPORTS_DIR as JSON immediately after each district finishes,
# instead of accumulating in a handful of ever-growing Python lists for
# the entire multi-hundred-district run. A district whose report already
# exists is skipped, so an interrupted run can be re-executed to resume.
# The Keras/TensorFlow session is cleared after every FOLD (not just once
# per district): every model built inside a fold is fully used (trained,
# predicted, metrics/history already extracted to plain floats) before
# the fold loop moves on, so clearing at that point cannot invalidate a
# model still needed later. None of this changes any computed value; it
# only changes when memory is freed and when partial results reach disk.
# ================================================================
duplicate_month_rows = panel_data.duplicated(["ADM2_CODE", "date"], keep=False)
ambiguous_districts = panel_data.loc[
    duplicate_month_rows, ["ADM2_CODE", "district", "province"]
].drop_duplicates().sort_values(["ADM2_CODE", "province", "district"])
ambiguous_districts.to_csv(os.path.join(OUTPUT_DIR, "excluded_ambiguous_district_codes.csv"), index=False)
ambiguous_ids = set(ambiguous_districts["ADM2_CODE"])
if ambiguous_ids:
    print(f"Excluded {len(ambiguous_ids)} ADM2_CODE values with duplicate district-month rows; see excluded_ambiguous_district_codes.csv")
all_district_ids = sorted(d for d in model_df["ADM2_CODE"].dropna().unique() if d not in ambiguous_ids)
if MAX_DISTRICTS is not None:
    all_district_ids = all_district_ids[:MAX_DISTRICTS]

# ----------------------------------------------------------------
# PROCESSING ORDER: run the districts that actually carry malaria
# burden first, and leave the near-zero ones until last.
#
# This is a SCHEDULING change only, and it is safe on both counts:
#   - Numbers: every district reseeds via seed_district(district_id)
#     and fits its own scalers, so a district's result
#     does not depend on how many districts ran before it. The
#     statistical models carry no RNG and XGBoost is seeded per call.
#   - Row order: the output tables are still assembled by loading
#     checkpoints in sorted all_district_ids order (Section R/S), NOT
#     in processing order, so every exported CSV keeps its original
#     layout.
#
# The benefit is failure tolerance: if a long run is interrupted or
# runs out of memory, the districts that carry the epidemiological
# signal are already checkpointed rather than still queued behind
# hundreds of all-zero series.
#
# LOW_BURDEN_THRESHOLD is total cases across the whole history. 0
# defers only districts with literally no cases; raise it to also
# defer very-low-burden districts.
# ----------------------------------------------------------------
LOW_BURDEN_THRESHOLD = 0

district_total_cases = model_df.groupby("ADM2_CODE")["target_cases"].sum()
high_burden_ids = [
    d for d in all_district_ids
    if float(district_total_cases.get(d, 0.0)) > LOW_BURDEN_THRESHOLD
]
low_burden_ids = [
    d for d in all_district_ids
    if float(district_total_cases.get(d, 0.0)) <= LOW_BURDEN_THRESHOLD
]
processing_order = high_burden_ids + low_burden_ids
print(
    f"Districts to process: {len(processing_order)} "
    f"({len(high_burden_ids)} above burden threshold first, "
    f"then {len(low_burden_ids)} at or below)"
)

# Computed once per notebook run. A district's checkpoint is only reused
# if it was written under this exact signature -- so a pilot run with a
# smaller MAX_DISTRICTS, or any change to the parameters below, can never
# be silently mixed into a later run with different settings.
RUN_SIGNATURE = compute_run_signature()
print("Run signature:", RUN_SIGNATURE)

def record_neural_history(history, district_id, district_name, province_name, fold_id, model_name, target_list):
    losses = history.history.get("loss", [])
    val_losses = history.history.get("val_loss", [np.nan] * len(losses))
    for epoch, loss_value in enumerate(losses, start=1):
        target_list.append([
            district_id, district_name, province_name, fold_id, model_name,
            epoch, float(loss_value), float(val_losses[epoch - 1]),
        ])


def run_validation_district(district_id):
    """Section Q for one district: five validation folds for all seven models,
    best-model selection, the 2025 external test, then one checkpoint file.
    Runs in this kernel or in a forked worker (see run_district_tasks)."""
    import tensorflow as tf  # imported in the worker, after fork (cell 1)
    if report_is_current(district_id, RUN_SIGNATURE):
        return

    district_df = model_df[model_df["ADM2_CODE"] == district_id].sort_values("date").copy()
    if district_df.shape[0] < 72:
        return

    # Reseed per district (not once globally) so every neural network this
    # district trains draws from the same weight-initialization stream
    # regardless of how many other districts were processed before it in
    # this session -- making a resumed or MAX_DISTRICTS-pilot-then-full run
    # produce the same neural-network result for this district as a single
    # uninterrupted run would. ARIMA/SARIMA/SARIMAX/XGBoost are unaffected;
    # they never depended on this global stream in the first place.
    # seed_district() uses keras.utils.set_random_seed by default: under
    # Keras 3, tf.random.set_seed does not drive weight initialisation or
    # dropout masks, so it did not make a district's neural results
    # repeatable. Set USE_KERAS_RANDOM_SEED = False in Section A to go back.
    seed_district(district_id)

    district_name = district_df["district"].iloc[0]
    province_name = district_df["province"].iloc[0]
    train_df = district_df[district_df["year"] <= TRAIN_END_YEAR].copy()
    test_df = district_df[district_df["year"] == TEST_YEAR].copy()
    validation_df = district_df[
        (district_df["year"] >= HIST_START_YEAR) & (district_df["year"] <= TRAIN_END_YEAR)
    ].copy()
    if len(train_df) < 60 or len(test_df) < TEST_HORIZON:
        return

    feature_cols_clean = dedup_keep_order([
        c for c in feature_cols
        if c in train_df.columns and c in validation_df.columns and c in test_df.columns
    ])
    if not feature_cols_clean:
        return

    y_train = train_df["target_cases"].values.astype(float)
    y_test = test_df["target_cases"].values.astype(float)
    exog_cols = [c for c in selected_env_features if c in train_df.columns and c in test_df.columns]
    if not exog_cols:
        exog_cols = [c for c in ["rain_mm_sum", "temp_c_mean", "RH_mean"] if c in train_df.columns and c in test_df.columns]

    fold_specs = build_expanding_yearly_folds(
        validation_df,
        start_year=HIST_START_YEAR,
        validation_years=EXPANDING_VALIDATION_YEARS,
    )
    if [fold["validation_year"] for fold in fold_specs] != EXPANDING_VALIDATION_YEARS:
        print(f"Incomplete validation calendar for district {district_id}; skipping.")
        return
    district_result_rows = []
    district_interval_rows = []
    district_parameter_rows = []
    district_forecast_rows = []
    district_neural_rows = []

    for fold_spec in fold_specs:
        fold_id = fold_spec["validation_year"]
        fold_train = validation_df[
            (validation_df["year"] >= fold_spec["train_start_year"]) &
            (validation_df["year"] <= fold_spec["train_end_year"])
        ].copy()
        fold_test = validation_df[validation_df["year"] == fold_spec["validation_year"]].copy()
        expected_train_months = MIN_FOLD_TRAIN_MONTHS + 12 * (fold_id - FIRST_VALIDATION_YEAR)
        if (len(fold_train) != expected_train_months or
                len(fold_test) != TEST_HORIZON or
                fold_test["date"].nunique() != TEST_HORIZON):
            raise ValueError(f"District {district_id}, fold {fold_id}: expected {expected_train_months} training months and {TEST_HORIZON} validation months.")

        # Fit predictor scaling within each fold to prevent information from
        # later validation years entering the fold-specific transformation.
        fold_scaler = MinMaxScaler()
        fold_train_X = fold_train[feature_cols_clean].apply(pd.to_numeric, errors="coerce").astype(float).fillna(0.0)
        fold_test_X = fold_test[feature_cols_clean].apply(pd.to_numeric, errors="coerce").astype(float).fillna(0.0)
        fold_train_scaled = pd.DataFrame(
            fold_scaler.fit_transform(fold_train_X),
            columns=feature_cols_clean,
            index=fold_train.index,
        )
        fold_test_scaled = pd.DataFrame(
            fold_scaler.transform(fold_test_X),
            columns=feature_cols_clean,
            index=fold_test.index,
        )
        fold_train = pd.concat([
            fold_train.drop(columns=feature_cols_clean, errors="ignore"), fold_train_scaled
        ], axis=1)
        fold_test = pd.concat([
            fold_test.drop(columns=feature_cols_clean, errors="ignore"), fold_test_scaled
        ], axis=1)

        y_tr = fold_train["target_cases"].values.astype(float)
        y_te = fold_test["target_cases"].values.astype(float)
        train_exog = fold_train[exog_cols].values.astype(float) if exog_cols else np.empty((len(fold_train), 0))
        test_exog = fold_test[exog_cols].values.astype(float) if exog_cols else np.empty((len(fold_test), 0))

        for model_name, fit_function in [
            ("ARIMA", lambda: fit_forecast_arima(y_tr, len(fold_test))),
            ("SARIMA", lambda: fit_forecast_sarima(y_tr, len(fold_test))),
            ("SARIMAX", lambda: fit_forecast_sarimax(y_tr, train_exog, test_exog)),
        ]:
            try:
                pred, conf, spec = fit_function()
                met = compute_metrics(y_te, pred)
                district_result_rows.append([
                    district_id, district_name, province_name, fold_id, model_name,
                    met["MAE"], met["RMSE"], met["sMAPE"],
                ])
                conf_array = np.asarray(conf)
                district_interval_rows.append([
                    district_id, district_name, province_name, fold_id, model_name,
                    float(np.nanmean(conf_array[:, 0])), float(np.nanmean(conf_array[:, 1])),
                ])
                district_parameter_rows.append([
                    district_id, district_name, province_name, fold_id, model_name,
                    str(spec[0]), str(spec[1]),
                    "AIC-selected" if model_name != "SARIMAX" else "AIC-selected with exogenous variables",
                    len(exog_cols) if model_name == "SARIMAX" else 0, np.nan, LOOK_BACK,
                ])
            except Exception as exc:
                print(f"{model_name} failed for {district_id}, fold {fold_id}: {exc}")

        if RUN_XGBOOST:
            try:
                pred, _, residual_sd = fit_forecast_xgboost(
                    y_tr, train_exog, test_exog, len(fold_test), look_back=LOOK_BACK
                )
                met = compute_metrics(y_te, pred)
                district_result_rows.append([
                    district_id, district_name, province_name, fold_id, "XGBoost",
                    met["MAE"], met["RMSE"], met["sMAPE"],
                ])
                district_interval_rows.append([
                    district_id, district_name, province_name, fold_id, "XGBoost",
                    float(np.mean(np.maximum(0.0, pred - 1.96 * residual_sd))),
                    float(np.mean(pred + 1.96 * residual_sd)),
                ])
                district_parameter_rows.append([
                    district_id, district_name, province_name, fold_id, "XGBoost",
                    str(XGB_PARAMS), "Recursive outcome lags", "Fixed regularized specification",
                    len(exog_cols), np.nan, LOOK_BACK,
                ])
            except Exception as exc:
                print(f"XGBoost failed for {district_id}, fold {fold_id}: {exc}")

        if USE_DEEP_LEARNING:
            try:
                scaler_y = MinMaxScaler()
                y_tr_scaled = scaler_y.fit_transform(y_tr.reshape(-1, 1)).reshape(-1)
                X_uni, y_uni = create_sequences_univariate(y_tr_scaled, look_back=LOOK_BACK)
                uni_model, uni_history, uni_params = train_tuned_lstm(X_uni, y_uni, "Univariate_LSTM")
                if uni_model is not None:
                    record_neural_history(uni_history, district_id, district_name, province_name, fold_id, "Univariate_LSTM", district_neural_rows)
                    pred = recursive_forecast_univariate(uni_model, y_tr_scaled, len(fold_test), scaler_y)
                    met = compute_metrics(y_te, pred)
                    district_result_rows.append([
                        district_id, district_name, province_name, fold_id, "Univariate_LSTM",
                        met["MAE"], met["RMSE"], met["sMAPE"],
                    ])
                    district_parameter_rows.append([
                        district_id, district_name, province_name, fold_id, "Univariate_LSTM",
                        str(uni_params), "Time-based validation", "Regularized candidate search",
                        0, len(uni_history.history.get("loss", [])), LOOK_BACK,
                    ])
            except Exception as exc:
                print(f"Univariate LSTM failed for {district_id}, fold {fold_id}: {exc}")

            mv_train = fold_train[["target_cases"] + feature_cols_clean].copy()
            scaler_mv_target = MinMaxScaler()
            mv_train["target_cases"] = scaler_mv_target.fit_transform(mv_train[["target_cases"]])
            X_mv, y_mv = create_sequences_multivariate(
                mv_train.values, mv_train["target_cases"].values, look_back=LOOK_BACK
            )
            # Unscaled validation-year features + observed cases up to the
            # fold origin; malaria lags are rebuilt from predictions.
            fold_lag_history = fold_train["malaria_cases"].values.astype(float)

            try:
                mv_model, mv_history, mv_params = train_tuned_lstm(X_mv, y_mv, "Multivariate_LSTM")
                if mv_model is not None:
                    record_neural_history(mv_history, district_id, district_name, province_name, fold_id, "Multivariate_LSTM", district_neural_rows)
                    pred = recursive_forecast_multivariate(
                        mv_model, mv_train.values, fold_test_X, fold_scaler,
                        scaler_mv_target, fold_lag_history,
                    )
                    met = compute_metrics(y_te, pred)
                    district_result_rows.append([
                        district_id, district_name, province_name, fold_id, "Multivariate_LSTM",
                        met["MAE"], met["RMSE"], met["sMAPE"],
                    ])
                    district_parameter_rows.append([
                        district_id, district_name, province_name, fold_id, "Multivariate_LSTM",
                        str(mv_params), "Time-based validation", "Regularized candidate search",
                        len(feature_cols_clean), len(mv_history.history.get("loss", [])), LOOK_BACK,
                    ])
            except Exception as exc:
                print(f"Multivariate LSTM failed for {district_id}, fold {fold_id}: {exc}")

            if RUN_GRU:
                try:
                    gru_model, gru_history, gru_params = train_compact_gru(X_mv, y_mv, "GRU")
                    if gru_model is not None:
                        record_neural_history(gru_history, district_id, district_name, province_name, fold_id, "GRU", district_neural_rows)
                        pred = recursive_forecast_multivariate(
                            gru_model, mv_train.values, fold_test_X, fold_scaler,
                            scaler_mv_target, fold_lag_history,
                        )
                        met = compute_metrics(y_te, pred)
                        district_result_rows.append([
                            district_id, district_name, province_name, fold_id, "GRU",
                            met["MAE"], met["RMSE"], met["sMAPE"],
                        ])
                        district_parameter_rows.append([
                            district_id, district_name, province_name, fold_id, "GRU",
                            str(gru_params), "Time-based validation", "Single compact specification",
                            len(feature_cols_clean), len(gru_history.history.get("loss", [])), LOOK_BACK,
                        ])
                except Exception as exc:
                    print(f"GRU failed for {district_id}, fold {fold_id}: {exc}")

        # Every model built in this fold has already been trained, used to
        # predict, and had its metrics/history extracted into plain numbers
        # above, so releasing the Keras session here cannot affect anything
        # still needed later.
        save_progress("district_validation_fold", district_id=district_id, fold_id=fold_id)
        tf.keras.backend.clear_session()
        gc.collect()

    if not district_result_rows:
        return

    district_results_df = pd.DataFrame(
        district_result_rows,
        columns=["ADM2_CODE", "district", "province", "fold", "model", "MAE", "RMSE", "sMAPE"],
    )
    complete_results = select_complete_validation_results(
        district_results_df, EXPANDING_VALIDATION_YEARS
    )
    if complete_results.empty:
        print(f"No model completed all five validation folds for district {district_id}.")
        save_district_report(
            district_id, district_result_rows, district_interval_rows, [],
            district_parameter_rows, district_neural_rows, RUN_SIGNATURE,
        )
        return
    mean_perf = (
        complete_results.groupby("model")[["MAE", "RMSE", "sMAPE"]]
        .mean().reset_index().sort_values(["RMSE", "MAE", "sMAPE"])
    )
    best_model = mean_perf.iloc[0]["model"]

    # Final predictor transformation is fitted only on 2016-2024 and then
    # applied to the external 2025 test period.
    final_scaler = MinMaxScaler()
    final_train_X = train_df[feature_cols_clean].apply(pd.to_numeric, errors="coerce").astype(float).fillna(0.0)
    final_test_X = test_df[feature_cols_clean].apply(pd.to_numeric, errors="coerce").astype(float).fillna(0.0)
    final_train_scaled = pd.DataFrame(
        final_scaler.fit_transform(final_train_X), columns=feature_cols_clean, index=train_df.index
    )
    final_test_scaled = pd.DataFrame(
        final_scaler.transform(final_test_X), columns=feature_cols_clean, index=test_df.index
    )
    train_df = pd.concat([
        train_df.drop(columns=feature_cols_clean, errors="ignore"), final_train_scaled
    ], axis=1)
    test_df = pd.concat([
        test_df.drop(columns=feature_cols_clean, errors="ignore"), final_test_scaled
    ], axis=1)

    lower = np.repeat(np.nan, len(test_df))
    upper = np.repeat(np.nan, len(test_df))
    try:
        train_exog = train_df[exog_cols].values.astype(float) if exog_cols else np.empty((len(train_df), 0))
        test_exog = test_df[exog_cols].values.astype(float) if exog_cols else np.empty((len(test_df), 0))
        if best_model == "ARIMA":
            pred, conf, _ = fit_forecast_arima(y_train, len(test_df))
            lower, upper = np.asarray(conf)[:, 0], np.asarray(conf)[:, 1]
        elif best_model == "SARIMA":
            pred, conf, _ = fit_forecast_sarima(y_train, len(test_df))
            lower, upper = np.asarray(conf)[:, 0], np.asarray(conf)[:, 1]
        elif best_model == "SARIMAX":
            pred, conf, _ = fit_forecast_sarimax(y_train, train_exog, test_exog)
            lower, upper = np.asarray(conf)[:, 0], np.asarray(conf)[:, 1]
        elif best_model == "XGBoost":
            pred, _, residual_sd = fit_forecast_xgboost(
                y_train, train_exog, test_exog, len(test_df), look_back=LOOK_BACK
            )
            lower = np.maximum(0.0, pred - 1.96 * residual_sd)
            upper = pred + 1.96 * residual_sd
        elif best_model == "Univariate_LSTM":
            scaler_y = MinMaxScaler()
            y_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).reshape(-1)
            X_uni, y_uni = create_sequences_univariate(y_scaled, look_back=LOOK_BACK)
            model, history, _ = train_tuned_lstm(X_uni, y_uni, "Univariate_LSTM final")
            if model is None:
                raise ValueError("Insufficient sequence length for Univariate_LSTM.")
            record_neural_history(history, district_id, district_name, province_name, "final_test", "Univariate_LSTM", district_neural_rows)
            pred = recursive_forecast_univariate(model, y_scaled, len(test_df), scaler_y)
            lower, upper = mc_dropout_interval(
                lambda n: forecast_paths_univariate(model, y_scaled, len(test_df), scaler_y, n_paths=n)
            )
        else:
            mv_train = train_df[["target_cases"] + feature_cols_clean].copy()
            scaler_target = MinMaxScaler()
            mv_train["target_cases"] = scaler_target.fit_transform(mv_train[["target_cases"]])
            X_mv, y_mv = create_sequences_multivariate(
                mv_train.values, mv_train["target_cases"].values, look_back=LOOK_BACK
            )
            final_lag_history = train_df["malaria_cases"].values.astype(float)
            if best_model == "Multivariate_LSTM":
                model, history, _ = train_tuned_lstm(X_mv, y_mv, "Multivariate_LSTM final")
            elif best_model == "GRU":
                model, history, _ = train_compact_gru(X_mv, y_mv, "GRU final")
            else:
                raise ValueError(f"Unsupported model: {best_model}")
            if model is None:
                raise ValueError(f"Insufficient sequence length for {best_model}.")
            record_neural_history(history, district_id, district_name, province_name, "final_test", best_model, district_neural_rows)
            pred = recursive_forecast_multivariate(
                model, mv_train.values, final_test_X, final_scaler,
                scaler_target, final_lag_history,
            )
            lower, upper = mc_dropout_interval(
                lambda n: forecast_paths_multivariate(
                    model, mv_train.values, final_test_X, final_scaler,
                    scaler_target, final_lag_history, n_paths=n,
                )
            )

        test_metrics = compute_metrics(y_test, pred)
        for i, row in test_df.reset_index(drop=True).iterrows():
            district_forecast_rows.append([
                district_id, district_name, province_name, row["date"], TEST_YEAR,
                int(row["month"]), best_model, float(y_test[i]), float(pred[i]),
                float(lower[i]) if i < len(lower) and pd.notnull(lower[i]) else np.nan,
                float(upper[i]) if i < len(upper) and pd.notnull(upper[i]) else np.nan,
                float(test_metrics["MAE"]), float(test_metrics["RMSE"]), float(test_metrics["sMAPE"]),
            ])
    except Exception as exc:
        print(f"Final test failed for {district_id} using {best_model}: {exc}")

    save_district_report(
        district_id,
        district_result_rows,
        district_interval_rows,
        district_forecast_rows,
        district_parameter_rows,
        district_neural_rows,
        RUN_SIGNATURE,
    )
    save_progress("district_validation", district_id=district_id)
    tf.keras.backend.clear_session()
    gc.collect()


# Districts whose checkpoint is already current are dropped here, in the
# parent, so the pool only receives work that still has to be done.
pending_validation = [d for d in processing_order if not report_is_current(d, RUN_SIGNATURE)]
run_district_tasks(run_validation_district, pending_validation, "Section Q")

mark_section_checkpoint(
    "Q_seven_model_runner",
    total_districts=len(all_district_ids),
    high_burden_districts=len(high_burden_ids),
    low_burden_districts=len(low_burden_ids),
    low_burden_threshold=LOW_BURDEN_THRESHOLD,
)

## SECTION R. VALIDATION RESULTS AND BEST MODEL TABLES (OUTPUT 2 / OBJECTIVE 2)

In [ ]:
# ================================================================
# SECTION R. VALIDATION RESULTS AND BEST MODEL TABLES (OUTPUT 2 / OBJECTIVE 2)

In [ ]:
# Read back the per-district JSON checkpoints written during Section Q
# instead of relying on Python lists that were held in RAM for the whole
# run. Loading in the original all_district_ids order keeps every row in
# the same order as the pre-checkpoint version of this notebook.
all_reports = load_district_reports(all_district_ids, RUN_SIGNATURE)

results_df = pd.DataFrame(
    [row for report in all_reports for row in report["results"]],
    columns=["ADM2_CODE", "district", "province", "fold", "model", "MAE", "RMSE", "sMAPE"],
)
interval_df = pd.DataFrame(
    [row for report in all_reports for row in report["interval"]],
    columns=["ADM2_CODE", "district", "province", "fold", "model", "lower_mean", "upper_mean"],
)
forecast_test_df = pd.DataFrame(
    [row for report in all_reports for row in report["forecast"]],
    columns=[
        "ADM2_CODE", "district", "province", "date", "year", "month", "best_model",
        "actual_cases", "predicted_cases", "lower_95", "upper_95", "MAE", "RMSE", "sMAPE",
    ],
)
if not forecast_test_df.empty:
    forecast_test_df["date"] = pd.to_datetime(forecast_test_df["date"])
parameter_df = pd.DataFrame(
    [row for report in all_reports for row in report["parameters"]],
    columns=[
        "ADM2_CODE", "district", "province", "fold", "model", "model_specification",
        "validation_specification", "selection_method", "n_exog", "n_neural_epochs", "lag_used",
    ],
)
neural_loss_history_df = pd.DataFrame(
    [row for report in all_reports for row in report["neural_loss_history"]],
    columns=["ADM2_CODE", "district", "province", "fold", "model", "epoch", "loss", "val_loss"],
)

if results_df.empty:
    raise ValueError("No model results were generated. Inspect the input fields and minimum time-series length.")

complete_results_df = select_complete_validation_results(
    results_df, EXPANDING_VALIDATION_YEARS
)
if complete_results_df.empty:
    raise ValueError("No district-model pair completed all five validation folds.")
complete_results_df.to_csv(
    os.path.join(OUTPUT_DIR, "complete_validation_results_all_models.csv"), index=False
)

model_summary = (
    complete_results_df.groupby("model")[["MAE", "RMSE", "sMAPE"]]
    .mean().reset_index().sort_values(["RMSE", "MAE", "sMAPE"])
)
best_model_df = (
    complete_results_df.groupby(["ADM2_CODE", "district", "province", "model"])[["MAE", "RMSE", "sMAPE"]]
    .mean().reset_index().sort_values(["ADM2_CODE", "RMSE", "MAE", "sMAPE"])
    .groupby("ADM2_CODE", as_index=False).first()
)

print("Validation results shape:", results_df.shape)
display(model_summary)
display(best_model_df.head())
mark_section_checkpoint("R_validation_results", n_district_reports=len(all_reports))

## SECTION S. FORECAST 2026 USING THE BEST MODEL FOR EACH DISTRICT
Explanation:

In [ ]:
# ================================================================
# SECTION S. FORECAST 2026 USING THE BEST MODEL FOR EACH DISTRICT
# Explanation:
# 2025 monthly climate is used as a proxy for 2026 if real climate forecasts are not available.

In [ ]:
# ================================================================
# SECTION S. FORECAST 2026 USING THE BEST MODEL FOR EACH DISTRICT
# Explanation:
# 2025 monthly climate is used as a proxy for 2026 if real climate forecasts are not available.
# This revised version also uses a safer recursive forecasting block so that
# climate variables, feature alignment, and scaling remain consistent.
# ================================================================
base_env_cols = ["rain_mm_sum", "temp_c_mean", "RH_mean"]
env_cols_for_proxy = dedup_keep_order(base_env_cols + selected_env_features)
env_cols_for_proxy = [c for c in env_cols_for_proxy if c in model_df.columns]

climate_proxy_2026 = (
    model_df.loc[model_df["year"] == TEST_YEAR, ["ADM2_CODE", "district", "province", "month"] + env_cols_for_proxy]
    .groupby(["ADM2_CODE", "district", "province", "month"], as_index=False)[env_cols_for_proxy]
    .mean()
)
climate_proxy_2026["year"] = FORECAST_YEAR
climate_proxy_2026 = add_cyclical_month_features(climate_proxy_2026)
if "kmeans_cluster" not in climate_proxy_2026.columns:
    climate_proxy_2026["kmeans_cluster"] = 0.0

def run_forecast_district(task):
    """Section S for one district: refit its best model on 2016-2025 and
    forecast 2026, then write one checkpoint file.
    Runs in this kernel or in a forked worker (see run_district_tasks)."""
    import tensorflow as tf  # imported in the worker, after fork (cell 1)
    district_id, best_model = task

    # Same memory-safe treatment as Section Q: this loop builds a fresh
    # LSTM/GRU per district too, so it needs the identical resume-skip,
    # per-district reseed, and post-use session clear.
    if forecast_report_is_current(district_id, RUN_SIGNATURE):
        return
    seed_district(district_id)  # same seeding rule as Section Q
    district_forecast_2026_rows = []

    district_df = model_df[model_df["ADM2_CODE"] == district_id].sort_values("date").copy()
    future_env = climate_proxy_2026[climate_proxy_2026["ADM2_CODE"] == district_id].sort_values("month").copy()

    if len(future_env) != 12 or district_df.empty:
        return

    district_name = district_df["district"].iloc[0]
    province_name = district_df["province"].iloc[0]

    future_env = ensure_columns(future_env, base_env_cols, fill_value=np.nan)
    district_df = ensure_columns(district_df, base_env_cols, fill_value=np.nan)
    district_df = safe_numeric_frame(district_df, base_env_cols, fill_value=np.nan)
    future_env = safe_numeric_frame(future_env, base_env_cols, fill_value=np.nan)

    for c in base_env_cols:
        if future_env[c].isna().any():
            monthly_fill_map = district_df.groupby("month")[c].mean().to_dict()
            future_env[c] = future_env.apply(
                lambda row: monthly_fill_map.get(row["month"], np.nan) if pd.isna(row[c]) else row[c],
                axis=1
            )
        if future_env[c].isna().any():
            future_env[c] = future_env[c].fillna(district_df[c].mean())
        future_env[c] = future_env[c].fillna(0.0)

    lag_cols = [
        "malaria_cases_lag1", "malaria_cases_lag2", "malaria_cases_lag3",
        "malaria_cases_rollmean_3", "malaria_cases_rollmean_6",
        "rain_mm_sum_lag1", "rain_mm_sum_lag2", "rain_mm_sum_lag3",
        "temp_c_mean_lag1", "temp_c_mean_lag2", "temp_c_mean_lag3",
        "RH_mean_lag1", "RH_mean_lag2", "RH_mean_lag3"
    ]
    future_env = ensure_columns(future_env, lag_cols, fill_value=np.nan)

    history_target = pd.to_numeric(district_df["target_cases"], errors="coerce").fillna(0.0).tolist()
    history_rain = pd.to_numeric(district_df["rain_mm_sum"], errors="coerce").fillna(0.0).tolist()
    history_temp = pd.to_numeric(district_df["temp_c_mean"], errors="coerce").fillna(0.0).tolist()
    history_rh = pd.to_numeric(district_df["RH_mean"], errors="coerce").fillna(0.0).tolist()

    while len(history_target) < 6:
        history_target.insert(0, history_target[0] if len(history_target) > 0 else 0.0)
    while len(history_rain) < 3:
        history_rain.insert(0, history_rain[0] if len(history_rain) > 0 else 0.0)
    while len(history_temp) < 3:
        history_temp.insert(0, history_temp[0] if len(history_temp) > 0 else 0.0)
    while len(history_rh) < 3:
        history_rh.insert(0, history_rh[0] if len(history_rh) > 0 else 0.0)

    # The malaria lag / rolling-mean values written below are placeholders:
    # Multivariate_LSTM and GRU, the only models that use them, rebuild them
    # from their own predictions at every step (forecast_paths_multivariate).
    for i in range(len(future_env)):
        idx = future_env.index[i]
        future_env.at[idx, "malaria_cases_lag1"] = float(history_target[-1])
        future_env.at[idx, "malaria_cases_lag2"] = float(history_target[-2])
        future_env.at[idx, "malaria_cases_lag3"] = float(history_target[-3])
        future_env.at[idx, "malaria_cases_rollmean_3"] = float(np.mean(history_target[-3:]))
        future_env.at[idx, "malaria_cases_rollmean_6"] = float(np.mean(history_target[-6:]))

        future_env.at[idx, "rain_mm_sum_lag1"] = float(history_rain[-1])
        future_env.at[idx, "rain_mm_sum_lag2"] = float(history_rain[-2])
        future_env.at[idx, "rain_mm_sum_lag3"] = float(history_rain[-3])
        future_env.at[idx, "temp_c_mean_lag1"] = float(history_temp[-1])
        future_env.at[idx, "temp_c_mean_lag2"] = float(history_temp[-2])
        future_env.at[idx, "temp_c_mean_lag3"] = float(history_temp[-3])
        future_env.at[idx, "RH_mean_lag1"] = float(history_rh[-1])
        future_env.at[idx, "RH_mean_lag2"] = float(history_rh[-2])
        future_env.at[idx, "RH_mean_lag3"] = float(history_rh[-3])

        history_rain.append(float(future_env.at[idx, "rain_mm_sum"]))
        history_temp.append(float(future_env.at[idx, "temp_c_mean"]))
        history_rh.append(float(future_env.at[idx, "RH_mean"]))
        history_target.append(float(history_target[-1]))

    feature_cols_clean = dedup_keep_order([c for c in feature_cols if c in district_df.columns])
    if len(feature_cols_clean) == 0:
        return
    future_env = ensure_columns(future_env, feature_cols_clean, fill_value=0.0)

    X_train = (district_df[feature_cols_clean]
               .apply(pd.to_numeric, errors="coerce")
               .astype(float)
               .fillna(0.0))
    X_future = (future_env[feature_cols_clean]
                .apply(pd.to_numeric, errors="coerce")
                .astype(float)
                .fillna(0.0))
    X_future = X_future.loc[:, X_train.columns]

    scaler_X = MinMaxScaler()
    X_train_scaled = pd.DataFrame(scaler_X.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_future_scaled = pd.DataFrame(scaler_X.transform(X_future), columns=X_train.columns, index=X_future.index)

    district_df_nonfeature = district_df.drop(columns=feature_cols_clean, errors="ignore").copy()
    district_df = pd.concat([district_df_nonfeature, X_train_scaled], axis=1)
    future_env_nonfeature = future_env.drop(columns=feature_cols_clean, errors="ignore").copy()
    future_env = pd.concat([future_env_nonfeature, X_future_scaled], axis=1)

    y_all = pd.to_numeric(district_df["target_cases"], errors="coerce").fillna(0.0).values.astype(float)
    lag_history_2026 = pd.to_numeric(district_df["malaria_cases"], errors="coerce").fillna(0.0).values.astype(float)
    exog_cols_safe = [c for c in selected_env_features if c in district_df.columns and c in future_env.columns]
    if not exog_cols_safe:
        exog_cols_safe = [c for c in base_env_cols if c in district_df.columns and c in future_env.columns]

    pred_2026 = np.repeat(np.nan, 12)
    lower = np.repeat(np.nan, 12)
    upper = np.repeat(np.nan, 12)

    try:
        if best_model == "ARIMA":
            pred_2026, conf, _ = fit_forecast_arima(y_all, 12)
            conf = np.asarray(conf)
            if conf.ndim == 2 and conf.shape[1] >= 2:
                lower, upper = conf[:, 0], conf[:, 1]
        elif best_model == "SARIMA":
            pred_2026, conf, _ = fit_forecast_sarima(y_all, 12)
            conf = np.asarray(conf)
            if conf.ndim == 2 and conf.shape[1] >= 2:
                lower, upper = conf[:, 0], conf[:, 1]
        elif best_model == "SARIMAX":
            if len(exog_cols_safe) == 0:
                raise ValueError("No aligned exogenous columns available for SARIMAX.")
            exog_all = district_df[exog_cols_safe].apply(pd.to_numeric, errors="coerce").astype(float).fillna(0.0).values
            exog_future = future_env[exog_cols_safe].apply(pd.to_numeric, errors="coerce").astype(float).fillna(0.0).values
            pred_2026, conf, _ = fit_forecast_sarimax(y_all, exog_all, exog_future)
            conf = np.asarray(conf)
            if conf.ndim == 2 and conf.shape[1] >= 2:
                lower, upper = conf[:, 0], conf[:, 1]

        elif best_model == "XGBoost":
            exog_all = district_df[exog_cols_safe].apply(pd.to_numeric, errors="coerce").astype(float).fillna(0.0).values
            exog_future = future_env[exog_cols_safe].apply(pd.to_numeric, errors="coerce").astype(float).fillna(0.0).values
            pred_2026, _, residual_sd = fit_forecast_xgboost(
                y_all, exog_all, exog_future, 12, look_back=LOOK_BACK
            )
            lower = np.maximum(0.0, pred_2026 - 1.96 * residual_sd)
            upper = pred_2026 + 1.96 * residual_sd
        elif best_model == "Univariate_LSTM":
            scaler_y = MinMaxScaler()
            y_scaled = scaler_y.fit_transform(y_all.reshape(-1, 1)).reshape(-1)
            X_uni, y_uni = create_sequences_univariate(y_scaled, look_back=LOOK_BACK)
            if len(X_uni) == 0:
                raise ValueError("Insufficient sequence length for Univariate_LSTM.")
            uni_model, uni_history, uni_params = train_tuned_lstm(X_uni, y_uni, "Univariate_LSTM 2026")
            if uni_model is None:
                raise ValueError("Insufficient sequence length for Univariate_LSTM.")
            pred_2026 = recursive_forecast_univariate(uni_model, y_scaled, 12, scaler_y)
            lower, upper = mc_dropout_interval(
                lambda n: forecast_paths_univariate(uni_model, y_scaled, 12, scaler_y, n_paths=n)
            )
        elif best_model == "Multivariate_LSTM":
            mv_train = pd.concat([
                pd.DataFrame({"target_cases": y_all}, index=district_df.index),
                X_train_scaled
            ], axis=1).copy()
            scaler_mv_target = MinMaxScaler()
            mv_train["target_cases"] = scaler_mv_target.fit_transform(mv_train[["target_cases"]])
            X_mv, y_mv = create_sequences_multivariate(mv_train.values, mv_train["target_cases"].values, look_back=LOOK_BACK)
            if len(X_mv) == 0:
                raise ValueError("Insufficient sequence length for Multivariate_LSTM.")
            mv_model, mv_history, mv_params = train_tuned_lstm(X_mv, y_mv, "Multivariate_LSTM 2026")
            if mv_model is None:
                raise ValueError("Insufficient sequence length for Multivariate_LSTM.")
            pred_2026 = recursive_forecast_multivariate(
                mv_model, mv_train.values, X_future, scaler_X, scaler_mv_target, lag_history_2026
            )
            lower, upper = mc_dropout_interval(
                lambda n: forecast_paths_multivariate(
                    mv_model, mv_train.values, X_future, scaler_X, scaler_mv_target,
                    lag_history_2026, n_paths=n,
                )
            )

        elif best_model == "GRU":
            mv_train = pd.concat([
                pd.DataFrame({"target_cases": y_all}, index=district_df.index),
                X_train_scaled,
            ], axis=1).copy()
            scaler_mv_target = MinMaxScaler()
            mv_train["target_cases"] = scaler_mv_target.fit_transform(mv_train[["target_cases"]])
            X_mv, y_mv = create_sequences_multivariate(
                mv_train.values, mv_train["target_cases"].values, look_back=LOOK_BACK
            )
            if len(X_mv) == 0:
                raise ValueError("Insufficient sequence length for GRU.")
            gru_model, _, _ = train_compact_gru(X_mv, y_mv, "GRU 2026")
            if gru_model is None:
                raise ValueError("Insufficient sequence length for GRU.")
            pred_2026 = recursive_forecast_multivariate(
                gru_model, mv_train.values, X_future, scaler_X, scaler_mv_target, lag_history_2026
            )
            lower, upper = mc_dropout_interval(
                lambda n: forecast_paths_multivariate(
                    gru_model, mv_train.values, X_future, scaler_X, scaler_mv_target,
                    lag_history_2026, n_paths=n,
                )
            )
        else:
            raise ValueError(f"Unsupported model name: {best_model}")
    except Exception as e:
        print(f"[ERROR] District {district_id} ({district_name}) | Model {best_model}: {e}")
        return

    pred_2026 = np.asarray(pred_2026).reshape(-1)
    if len(pred_2026) != 12:
        return

    for i in range(12):
        district_forecast_2026_rows.append([
            district_id, district_name, province_name, FORECAST_YEAR, i + 1,
            best_model, float(pred_2026[i]) if pd.notnull(pred_2026[i]) else np.nan,
            float(lower[i]) if i < len(lower) and pd.notnull(lower[i]) else np.nan,
            float(upper[i]) if i < len(upper) and pd.notnull(upper[i]) else np.nan
        ])

    # Every model built above has already been used for prediction and had
    # its output extracted into plain floats, so the checkpoint write and
    # session clear below are safe and this district is done.
    save_forecast_report(district_id, district_forecast_2026_rows, RUN_SIGNATURE)
    tf.keras.backend.clear_session()
    gc.collect()

pending_forecast = [
    (br["ADM2_CODE"], br["model"]) for _, br in best_model_df.iterrows()
    if not forecast_report_is_current(br["ADM2_CODE"], RUN_SIGNATURE)
]
run_district_tasks(run_forecast_district, pending_forecast, "Section S")

# Read back the per-district JSON checkpoints (written above, or in an
# earlier run under the same RUN_SIGNATURE) instead of relying on a Python
# list held in RAM for the whole loop, and in the original all_district_ids
# order so row order matches the pre-checkpoint version of this notebook.
forecast_2026_reports = load_forecast_reports(all_district_ids, RUN_SIGNATURE)
forecast_2026_df = pd.DataFrame(
    [row for report in forecast_2026_reports for row in report["forecast"]],
    columns=[
        "ADM2_CODE", "district", "province", "year", "month", "best_model",
        "forecast_cases_2026", "lower_95", "upper_95"
    ],
)

forecast_2026_df = forecast_2026_df.sort_values(["ADM2_CODE", "year", "month"]).reset_index(drop=True)
print("2026 forecast shape:", forecast_2026_df.shape)
display(forecast_2026_df.head())
mark_section_checkpoint("S_forecast_2026", n_district_reports=len(forecast_2026_reports))

## SECTION T. IDENTIFY RISK DISTRICTS (OUTPUT 3 / OBJECTIVE 3)
Explanation:

In [ ]:
# ================================================================
# SECTION T. IDENTIFY RISK DISTRICTS (OUTPUT 3 / OBJECTIVE 3)
# Explanation:
# High-risk if forecast 2026 annual burden > district median annual burden over the most recent 5 years (2021-2025)
# Low-risk otherwise.

In [ ]:
# ================================================================
forecast_summary = (forecast_2026_df.groupby(["ADM2_CODE", "district", "province", "best_model"], as_index=False)["forecast_cases_2026"].sum())

recent_historical = model_df[(model_df["year"] >= RECENT_MEDIAN_START) & (model_df["year"] <= HIST_END_YEAR)].copy()
recent_annual = recent_historical.groupby(["ADM2_CODE", "year"], as_index=False)["target_cases"].sum()
recent_annual_median = (recent_annual.groupby("ADM2_CODE", as_index=False)["target_cases"]
                        .median()
                        .rename(columns={"target_cases": "recent_5yr_annual_median"}))

risk_df = forecast_summary.merge(recent_annual_median, on="ADM2_CODE", how="left")
risk_df["risk_level"] = np.where(
    risk_df["forecast_cases_2026"] > risk_df["recent_5yr_annual_median"],
    "High risk",
    "Low risk"
)

risk_count = risk_df["risk_level"].value_counts().reset_index()
risk_count.columns = ["risk_level", "district_count"]

display(risk_df.head())
display(risk_count)
mark_section_checkpoint("T_risk_districts")

## SECTION U. EXPORT OUTPUTS

In [ ]:
# ================================================================
# SECTION U. EXPORT OUTPUTS

In [ ]:
# ================================================================
sex_table.to_csv(f"{OUTPUT_DIR}/characteristics_sex_2016_2025.csv", index=False)
age_table.to_csv(f"{OUTPUT_DIR}/characteristics_age_2016_2025.csv", index=False)
year_table.to_csv(f"{OUTPUT_DIR}/characteristics_year_2016_2025.csv", index=False)
province_table.to_csv(f"{OUTPUT_DIR}/characteristics_province_2016_2025.csv", index=False)
panel_data.to_csv(f"{OUTPUT_DIR}/integrated_monthly_district_panel_2016_2025.csv", index=False)
model_df.to_csv(f"{OUTPUT_DIR}/modeling_dataset_process_aligned_2016_2025.csv", index=False)
outlier_bounds_df.to_csv(f"{OUTPUT_DIR}/outlier_bounds_by_district_2016_2025.csv", index=False)
results_df.to_csv(f"{OUTPUT_DIR}/rolling_validation_results_all_models.csv", index=False)
interval_df.to_csv(f"{OUTPUT_DIR}/uncertainty_interval_summary_validation.csv", index=False)
model_summary.to_csv(f"{OUTPUT_DIR}/model_summary_mean_performance.csv", index=False)
parameter_df.to_csv(f"{OUTPUT_DIR}/model_parameter_selection_summary.csv", index=False)
best_model_df.to_csv(f"{OUTPUT_DIR}/best_model_by_district.csv", index=False)
forecast_test_df.to_csv(f"{OUTPUT_DIR}/test_predictions_2025_best_model.csv", index=False)
forecast_2026_df.to_csv(f"{OUTPUT_DIR}/forecast_2026_by_district_month.csv", index=False)
risk_df.to_csv(f"{OUTPUT_DIR}/risk_districts_2026_vs_recent5yrmedian.csv", index=False)
risk_count.to_csv(f"{OUTPUT_DIR}/risk_count_2026.csv", index=False)

print("All files exported to:", OUTPUT_DIR)
mark_section_checkpoint("U_export_outputs")

# ADDITIONAL THESIS-LEVEL RESULTS AND VISUALIZATION SECTIONS

The following sections produce the requested tables, line graphs, map-ready outputs, validation graphics, and uncertainty summaries. They are designed to run after the main `RunModel20` workflow has completed.

## SECTION V. DATA COLLECTION RESULTS: CLIMATE LINE GRAPHS AND MAP-READY CLIMATE OUTPUTS

This section summarizes Thailand's climate dataset using monthly line graphs for rainfall, temperature, and relative humidity. If latitude and longitude are available in the climate file, it also creates spatial scatter maps as a practical map-ready visualization. A combined figure (`climate_line_combined.png`) stacks the three series in one image, one panel per variable with its own y-axis, sharing the date axis.


In [ ]:
# ================================================================
# SECTION V. DATA COLLECTION RESULTS: CLIMATE LINE GRAPHS AND MAP-READY CLIMATE OUTPUTS
# ================================================================
climate_plot_df = climate.copy()
climate_plot_df["date"] = pd.to_datetime(dict(year=climate_plot_df["year"], month=climate_plot_df["month"], day=1))

climate_monthly_summary = (
    climate_plot_df.groupby("date", as_index=False)[["rain_mm_sum", "temp_c_mean", "RH_mean"]]
    .mean()
)

for var, ylabel, filename in [
    ("rain_mm_sum", "Mean rainfall (mm)", "climate_line_rainfall.png"),
    ("temp_c_mean", "Mean temperature (°C)", "climate_line_temperature.png"),
    ("RH_mean", "Mean relative humidity (%)", "climate_line_relative_humidity.png"),
]:
    plt.figure(figsize=(12, 5))
    plt.plot(climate_monthly_summary["date"], climate_monthly_summary[var])
    plt.title(f"Monthly Thailand Climate Pattern: {ylabel}, {HIST_START_YEAR}-{HIST_END_YEAR}")
    plt.xlabel("Date")
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{filename}", dpi=300)
    plt.show()

# Combined figure: the three climate series in one image, one panel per
# variable (each keeps its own y-axis and unit), sharing the date axis.
import matplotlib.dates as mdates

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, (var, ylabel, color) in zip(axes, [
    ("rain_mm_sum", "Rainfall (mm)", "#1f77b4"),
    ("temp_c_mean", "Temperature (°C)", "#d62728"),
    ("RH_mean", "Relative humidity (%)", "#2ca02c"),
]):
    ax.plot(climate_monthly_summary["date"], climate_monthly_summary[var], color=color, linewidth=1.3)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.suptitle(f"Monthly Thailand Climate Pattern, {HIST_START_YEAR}-{HIST_END_YEAR}", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/climate_line_combined.png", dpi=300)
fig.savefig(f"{OUTPUT_DIR}/climate_line_combined.pdf")
plt.show()

climate_district_summary = (
    climate_plot_df.groupby(["ADM2_CODE", "district", "province"], as_index=False)
    [["rain_mm_sum", "temp_c_mean", "RH_mean"]]
    .mean()
)

# Detect possible latitude/longitude columns for map-ready climate visualization.
lat_candidates = ["lat", "latitude", "Latitude", "LAT", "centroid_lat", "y"]
lon_candidates = ["lon", "longitude", "Longitude", "LON", "centroid_lon", "x"]
lat_col = next((c for c in lat_candidates if c in climate_plot_df.columns), None)
lon_col = next((c for c in lon_candidates if c in climate_plot_df.columns), None)

if lat_col and lon_col:
    coords = climate_plot_df.groupby("ADM2_CODE", as_index=False)[[lat_col, lon_col]].mean()
    climate_district_summary = climate_district_summary.merge(coords, on="ADM2_CODE", how="left")
    for var, title, filename in [
        ("rain_mm_sum", "Mean Rainfall by District", "map_climate_rainfall_scatter.png"),
        ("temp_c_mean", "Mean Temperature by District", "map_climate_temperature_scatter.png"),
        ("RH_mean", "Mean Relative Humidity by District", "map_climate_rh_scatter.png"),
    ]:
        plt.figure(figsize=(7, 9))
        sc = plt.scatter(climate_district_summary[lon_col], climate_district_summary[lat_col], c=climate_district_summary[var], s=12)
        plt.colorbar(sc, label=var)
        plt.title(f"{title}, {HIST_START_YEAR}-{HIST_END_YEAR}")
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/{filename}", dpi=300)
        plt.show()
else:
    print("Latitude/longitude columns were not found. A map-ready climate summary CSV will be exported instead.")

climate_monthly_summary.to_csv(f"{OUTPUT_DIR}/climate_monthly_summary_for_line_graphs.csv", index=False)
climate_district_summary.to_csv(f"{OUTPUT_DIR}/climate_district_summary_map_ready.csv", index=False)
display(climate_monthly_summary.head())
display(climate_district_summary.head())
mark_section_checkpoint("V_climate_graphs")

## SECTION W. DATA INTEGRATION AND PRELIMINARY PROCESSING RESULTS

This section creates compact tables that can be copied directly into the thesis results chapter.

In [ ]:
# ================================================================
# SECTION W. DATA INTEGRATION AND PRELIMINARY PROCESSING RESULTS
# ================================================================
integration_summary = pd.DataFrame([
    ["Raw malaria data", malaria_raw.shape[0], malaria_raw.shape[1]],
    ["Raw climate data", climate_raw.shape[0], climate_raw.shape[1]],
    ["Monthly malaria data", malaria_monthly.shape[0], malaria_monthly.shape[1]],
    ["Prepared climate data", climate.shape[0], climate.shape[1]],
    ["Integrated district-month panel", panel_data.shape[0], panel_data.shape[1]],
    ["Modeling dataset after preprocessing", model_df.shape[0], model_df.shape[1]],
], columns=["Dataset", "Number_of_records", "Number_of_variables"])

preprocessing_summary = pd.DataFrame([
    ["KNN missing-value imputation", "Climate and intervention predictors", f"KNNImputer(n_neighbors={SELECTED_KNN_K}); validated over {KNN_CANDIDATES}", int(model_df[imputer_cols].isna().sum().sum()) if imputer_cols else 0],
    ["Feature engineering", "Lag, rolling mean, cyclic month", f"Lag 1-{LOOK_BACK}; rolling 3 and 6 months; sin/cos month", len(feature_cols)],
    ["Clustering", "Environmental grouping", "KMeans(n_clusters=4)", model_df["kmeans_cluster"].nunique() if "kmeans_cluster" in model_df.columns else np.nan],
    ["Outlier management", "District-level malaria cases", "IQR capping per district", int(model_df["malaria_cases_outlier"].sum()) if "malaria_cases_outlier" in model_df.columns else np.nan],
    ["Normalization", "Model predictors", "MinMaxScaler fitted on training data only", len(feature_cols)],
], columns=["Process", "Target_variables", "Method", "Result_value"])

display(integration_summary)
display(preprocessing_summary)
integration_summary.to_csv(f"{OUTPUT_DIR}/integration_summary.csv", index=False)
preprocessing_summary.to_csv(f"{OUTPUT_DIR}/preprocessing_summary.csv", index=False)
mark_section_checkpoint("W_integration_summary")

## SECTION X. MODEL DEVELOPMENT RESULTS: TRAIN/TEST SPLIT AND PARAMETER SELECTION

This section reports the number of records used for model development and summarizes model specifications selected during rolling-origin validation.

In [ ]:
# ================================================================
# SECTION X. MODEL DEVELOPMENT RESULTS: TRAIN/TEST SPLIT AND PARAMETER SELECTION
# ================================================================
train_test_summary = pd.DataFrame([
    ["Training dataset", f"{HIST_START_YEAR}-{TRAIN_END_YEAR}", train_data.shape[0], train_data["ADM2_CODE"].nunique(), train_data.shape[1]],
    ["Test dataset", str(TEST_YEAR), test_data.shape[0], test_data["ADM2_CODE"].nunique(), test_data.shape[1]],
], columns=["Dataset", "Period", "Records", "Districts", "Variables"])

display(train_test_summary)

if "parameter_df" in globals() and not parameter_df.empty:
    # parameter_df has no 'order'/'seasonal_order' columns -- it never did.
    # SECTION Q writes the model's own specification into column 6 and the
    # validation description into column 7, and SECTION R names them
    # 'model_specification' and 'validation_specification' because their
    # content is model-dependent: an ARIMA order tuple for the statistical
    # models, but a hyper-parameter dict for LSTM/GRU and the XGBoost
    # parameter dict for XGBoost. Grouping by the old names raised KeyError.
    # This bug was latent until now: every earlier run died inside SECTION Q
    # and never reached this section.
    parameter_selection_summary = (
        parameter_df.groupby(
            ["model", "model_specification", "validation_specification", "selection_method"],
            dropna=False,
        )
        .size()
        .reset_index(name="Frequency")
        .sort_values(["model", "Frequency"], ascending=[True, False])
    )
else:
    parameter_selection_summary = pd.DataFrame({"Note": ["Parameter rows were not generated. Re-run SECTION Q after the enhanced notebook patch."]})

display(parameter_selection_summary.head(30))
train_test_summary.to_csv(f"{OUTPUT_DIR}/train_test_split_summary.csv", index=False)
parameter_selection_summary.to_csv(f"{OUTPUT_DIR}/parameter_selection_summary_frequency.csv", index=False)
mark_section_checkpoint("X_train_test_parameter_summary")

## SECTION Y. MODEL VALIDATION RESULTS: INDICATOR GRAPHS, NEURAL LOSS CURVES, PREDICTED VERSUS OBSERVED VALUES, AND UNCERTAINTY

This section creates publication-ready MAE, RMSE, and sMAPE comparisons for all seven models; training and validation loss curves for LSTM and GRU models; predicted-versus-observed plots; final-test error summaries; and uncertainty summaries where available.


In [ ]:
# ================================================================
# SECTION Y. MODEL VALIDATION RESULTS
# Publication-ready validation figures, loss curves, predicted vs target,
# and uncertainty summaries
# ================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

VALIDATION_PLOT_DIR = os.path.join(OUTPUT_DIR, "publication_validation_results")
os.makedirs(VALIDATION_PLOT_DIR, exist_ok=True)

MODEL_ORDER = [
    "SARIMAX",
    "SARIMA",
    "ARIMA",
    "Univariate_LSTM",
    "Multivariate_LSTM",
    "GRU",
    "XGBoost"
]

MODEL_COLORS = {
    "SARIMAX": "#1f77b4",
    "SARIMA": "#2ca02c",
    "ARIMA": "#ff7f0e",
    "Univariate_LSTM": "#9467bd",
    "Multivariate_LSTM": "#d62728",
    "GRU": "#17becf",
    "XGBoost": "#8c564b"
}

# ================================================================
# Helper functions
# ================================================================

def save_current_figure(filename):
    png_path = os.path.join(VALIDATION_PLOT_DIR, f"{filename}.png")
    pdf_path = os.path.join(VALIDATION_PLOT_DIR, f"{filename}.pdf")
    plt.tight_layout()
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.show()
    print(f"Saved PNG: {png_path}")
    print(f"Saved PDF: {pdf_path}")


def add_bar_labels(ax, bars, values, indicator):
    abs_max = max(np.nanmax(np.abs(values)), 1e-9)
    for bar, value in zip(bars, values):
        if pd.isna(value):
            continue
        label = f"{value:.3f}" if indicator in ["MAE", "RMSE"] else f"{value:.2f}"
        if value >= 0:
            y_pos = value + abs_max * 0.025
            va = "bottom"
        else:
            y_pos = value - abs_max * 0.035
            va = "top"
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            y_pos,
            label,
            ha="center",
            va=va,
            fontsize=10,
            fontweight="bold"
        )


def smooth_series(values, window=3):
    return (
        pd.Series(values, dtype="float")
        .rolling(window=window, min_periods=1, center=True)
        .mean()
        .values
    )

# ================================================================
# A) Publication-ready validation indicator bar charts
# ================================================================

indicator_cols = [c for c in ["MAE", "RMSE", "sMAPE"] if c in results_df.columns]

validation_long = results_df.melt(
    id_vars=["ADM2_CODE", "district", "province", "fold", "model"],
    value_vars=indicator_cols,
    var_name="Indicator",
    value_name="Value"
)

for indicator in indicator_cols:
    plot_df = (
        results_df
        .groupby("model", as_index=False)[indicator]
        .mean()
        .dropna()
    )

    # Lower values indicate better performance for all retained indicators.
    plot_df = plot_df.sort_values(indicator, ascending=True)

    colors = [MODEL_COLORS.get(m, "#7f7f7f") for m in plot_df["model"]]

    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    bars = ax.bar(
        plot_df["model"],
        plot_df[indicator],
        color=colors,
        edgecolor="black",
        linewidth=0.6
    )

    add_bar_labels(ax, bars, plot_df[indicator].values, indicator)

    ax.set_title(f"Mean Validation {indicator} by Model", fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(indicator, fontsize=12)
    ax.tick_params(axis="x", labelrotation=25)
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ymin = plot_df[indicator].min()
    ymax = plot_df[indicator].max()
    abs_max = max(abs(ymin), abs(ymax), 1e-9)

    if ymin < 0:
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_ylim(ymin - abs_max * 0.15, max(0.1, ymax + abs_max * 0.15))
    else:
        ax.set_ylim(0, ymax + abs_max * 0.18)

    save_current_figure(f"validation_indicator_{indicator}_publication")

# ================================================================
# B) LSTM loss curves by epoch
# ================================================================
# Note: ARIMA/SARIMA/SARIMAX/XGBoost do not train by epochs. Therefore, epoch-level
# loss curves are produced for the LSTM and GRU models. Classical models are evaluated
# using validation error indicators and forecast intervals.

if "neural_loss_history_df" in globals() and not neural_loss_history_df.empty:
    loss_summary_df = (
        neural_loss_history_df
        .groupby(["model", "epoch"], as_index=False)
        .agg(
            loss=("loss", "mean"),
            val_loss=("val_loss", "mean")
        )
        .sort_values(["model", "epoch"])
    )

    for model_name in ["Univariate_LSTM", "Multivariate_LSTM", "GRU"]:
        model_loss = loss_summary_df[loss_summary_df["model"] == model_name].copy()
        if model_loss.empty:
            print(f"No loss history available for {model_name}")
            continue

        model_loss["loss_smooth"] = smooth_series(model_loss["loss"], window=3)
        model_loss["val_loss_smooth"] = smooth_series(model_loss["val_loss"], window=3)

        fig, ax = plt.subplots(figsize=(9.5, 4.8))
        ax.plot(
            model_loss["epoch"],
            model_loss["loss_smooth"],
            label="Training dataset",
            color="#1f77b4",
            linewidth=2.2
        )
        ax.plot(
            model_loss["epoch"],
            model_loss["val_loss_smooth"],
            label="Validation dataset",
            color="#ff7f0e",
            linewidth=2.2
        )

        ax.set_title(f"Training and Validation Loss by Epoch ({model_name})", fontsize=14, fontweight="bold")
        ax.set_xlabel("Epoch", fontsize=12)
        ax.set_ylabel("Loss", fontsize=12)
        ax.grid(True, linestyle="--", alpha=0.35)
        ax.legend(frameon=True, fontsize=10)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        save_current_figure(f"loss_curve_{model_name}_publication")

    loss_summary_df.to_csv(
        os.path.join(OUTPUT_DIR, "neural_epoch_loss_summary.csv"),
        index=False
    )
else:
    print("No LSTM loss history table was found. Re-run Section Q after the revised LSTM fit code to collect epoch-level loss and validation loss.")

# ================================================================
# C) Loss/error summary table for the final test dataset
# ================================================================

loss_rows = []
for model_name, g in forecast_test_df.groupby("best_model"):
    y_true = g["actual_cases"].astype(float).values
    y_pred = g["predicted_cases"].astype(float).values
    loss_rows.append({
        "Model": model_name,
        "Test_MSE_or_loss": np.mean((y_true - y_pred) ** 2),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "sMAPE": smape(y_true, y_pred)
    })

loss_error_summary = pd.DataFrame(loss_rows)
display(loss_error_summary)

# ================================================================
# D) Prepare final test prediction table for smoothed monthly plots
# ================================================================

all_final_test_predictions = forecast_test_df.rename(
    columns={
        "best_model": "model",
        "actual_cases": "actual",
        "predicted_cases": "predicted"
    }
)[["model", "date", "actual", "predicted"]].copy()

all_final_test_predictions["date"] = pd.to_datetime(all_final_test_predictions["date"])


def prepare_complete_monthly_prediction(df, model_name):
    temp = df[df["model"] == model_name].copy()
    if temp.empty:
        return pd.DataFrame()

    temp["month_date"] = temp["date"].dt.to_period("M").dt.to_timestamp()

    monthly_df = (
        temp
        .groupby("month_date", as_index=False)
        .agg(
            actual=("actual", "mean"),
            predicted=("predicted", "mean")
        )
        .sort_values("month_date")
    )

    year_value = int(monthly_df["month_date"].dt.year.mode().iloc[0])
    complete_months = pd.date_range(
        start=f"{year_value}-01-01",
        end=f"{year_value}-12-01",
        freq="MS"
    )

    monthly_df = pd.DataFrame({"month_date": complete_months}).merge(
        monthly_df,
        on="month_date",
        how="left"
    )

    monthly_df["actual"] = monthly_df["actual"].interpolate().ffill().bfill()
    monthly_df["predicted"] = monthly_df["predicted"].interpolate().ffill().bfill()
    monthly_df["actual_smooth"] = smooth_series(monthly_df["actual"], window=3)
    monthly_df["predicted_smooth"] = smooth_series(monthly_df["predicted"], window=3)

    return monthly_df

# ================================================================
# E) Smoothed predicted vs target plots with complete 12-month X-axis
# ================================================================

for model_name in MODEL_ORDER:
    monthly_df = prepare_complete_monthly_prediction(all_final_test_predictions, model_name)

    if monthly_df.empty:
        print(f"No final test predictions available for {model_name}. This can occur if no district selected this model as the best model.")
        continue

    fig, ax = plt.subplots(figsize=(9.5, 4.8))

    ax.plot(
        monthly_df["month_date"],
        monthly_df["actual_smooth"],
        label="Target",
        color="#1f77b4",
        marker="o",
        linewidth=2.2
    )
    ax.plot(
        monthly_df["month_date"],
        monthly_df["predicted_smooth"],
        label="Predicted",
        color="#ff7f0e",
        marker="s",
        linestyle="--",
        linewidth=2.2
    )

    ax.set_title(f"Predicted vs Target: Final Test Dataset ({model_name})", fontsize=14, fontweight="bold")
    ax.set_xlabel("Month", fontsize=12)
    ax.set_ylabel("Mean malaria cases", fontsize=12)

    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.legend(frameon=True, fontsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    save_current_figure(f"smoothed_predicted_vs_target_final_test_{model_name}")

# ================================================================
# F) Overall predicted vs target scatter plot
# ================================================================

plt.figure(figsize=(6, 6))
plt.scatter(
    forecast_test_df["actual_cases"],
    forecast_test_df["predicted_cases"],
    alpha=0.45,
    edgecolor="none"
)
max_val = np.nanmax([
    forecast_test_df["actual_cases"].max(),
    forecast_test_df["predicted_cases"].max()
])
plt.plot([0, max_val], [0, max_val], linestyle="--", color="black", linewidth=1)
plt.title("Predicted vs Target Malaria Cases: Final Test Dataset", fontsize=13, fontweight="bold")
plt.xlabel("Observed target cases")
plt.ylabel("Predicted cases")
plt.grid(True, linestyle="--", alpha=0.30)
save_current_figure("predicted_vs_target_scatter_final_test_all_models")

# ================================================================
# G) Uncertainty analysis summary
# ================================================================

if {"lower_95", "upper_95"}.issubset(forecast_test_df.columns):
    forecast_test_df["interval_width_95"] = forecast_test_df["upper_95"] - forecast_test_df["lower_95"]
    # coverage_95: share of 2025 months whose observed (capped) target lies
    # inside the interval; a well-calibrated 95% interval gives about 0.95.
    interval_available = forecast_test_df[["lower_95", "upper_95"]].notna().all(axis=1)
    forecast_test_df["inside_95"] = np.where(
        interval_available,
        ((forecast_test_df["actual_cases"] >= forecast_test_df["lower_95"]) &
         (forecast_test_df["actual_cases"] <= forecast_test_df["upper_95"])).astype(float),
        np.nan,
    )

    uncertainty_summary = (
        forecast_test_df
        .groupby("best_model", as_index=False)
        .agg(
            mean_lower_95=("lower_95", "mean"),
            mean_upper_95=("upper_95", "mean"),
            mean_interval_width_95=("interval_width_95", "mean"),
            median_interval_width_95=("interval_width_95", "median"),
            n_months_with_interval=("interval_width_95", "count"),
            coverage_95=("inside_95", "mean"),
        )
    )

    display(uncertainty_summary)
else:
    uncertainty_summary = pd.DataFrame()
    print("Uncertainty columns lower_95 and upper_95 were not found.")

# ================================================================
# H) Export tables
# ================================================================

validation_long.to_csv(os.path.join(OUTPUT_DIR, "validation_indicators_long.csv"), index=False)
loss_error_summary.to_csv(os.path.join(OUTPUT_DIR, "loss_error_summary_test_dataset.csv"), index=False)
all_final_test_predictions.to_csv(os.path.join(OUTPUT_DIR, "final_test_predictions_for_plotting.csv"), index=False)

if not uncertainty_summary.empty:
    uncertainty_summary.to_csv(os.path.join(OUTPUT_DIR, "uncertainty_summary_by_best_model.csv"), index=False)

print("SECTION Y completed successfully.")
mark_section_checkpoint("Y_validation_plots")

### Train, external test, and predicted series by selected model

The plots show the mean monthly case count among the same districts that selected each model. The training line ends in 2024; the observed test and forecast lines cover 2025. Plots and monthly values are saved under `train_test_predicted_by_model` in the current output directory.


In [ ]:
# ================================================================
# SECTION Y. TRAIN, TEST AND PREDICTED TIMELINES BY SELECTED MODEL
# make_plots() below was the runmodel21_audit/plot_model_timelines.py
# helper module; it is included here so the notebook is self-contained.
# ================================================================
from pathlib import Path

import matplotlib

# The helper module switched to the non-interactive Agg backend on import;
# kept so the figures and outputs match the earlier runs.
matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt


TIMELINE_MODEL_ORDER = (
    "ARIMA", "SARIMA", "SARIMAX", "Univariate_LSTM",
    "Multivariate_LSTM", "GRU", "XGBoost",
)


def make_plots(input_dir: Path, output_dir: Path) -> pd.DataFrame:
    predictions = pd.read_csv(
        input_dir / "test_predictions_2025_best_model.csv",
        dtype={"ADM2_CODE": "string"},
        parse_dates=["date"],
    )
    history = pd.read_csv(
        input_dir / "modeling_dataset_process_aligned_2016_2025.csv",
        dtype={"ADM2_CODE": "string"},
        usecols=["ADM2_CODE", "date", "target_cases"],
        parse_dates=["date"],
    )

    if predictions.empty or history.empty:
        raise ValueError("Prediction or training dataset is empty")
    if predictions["date"].isna().any() or history["date"].isna().any():
        raise ValueError("A date could not be parsed")
    if predictions.groupby("ADM2_CODE")["best_model"].nunique().gt(1).any():
        raise ValueError("A district has more than one selected model")
    duplicate_pairs = int(predictions.duplicated(["ADM2_CODE", "date"]).sum())
    if duplicate_pairs:
        print(f"Found {duplicate_pairs} extra test rows; summing within district-month")
    predictions = predictions.groupby(["ADM2_CODE", "date", "best_model"], as_index=False).agg(
        actual_cases=("actual_cases", "sum"),
        predicted_cases=("predicted_cases", "sum"),
    )

    test_years = predictions["date"].dt.year.unique()
    if len(test_years) != 1 or int(test_years[0]) != 2025:
        raise ValueError(f"Expected external-test predictions only for 2025, got {test_years}")
    expected_test_dates = pd.date_range("2025-01-01", "2025-12-01", freq="MS")
    output_dir.mkdir(parents=True, exist_ok=True)

    # Older notebook runs can contain multiple rows per district-month. Aggregate
    # their case counts before taking the district mean; never double-weight an ID.
    history = history.loc[history["date"] < expected_test_dates[0]].copy()
    history_by_district = history.groupby(["ADM2_CODE", "date"], as_index=False)["target_cases"].sum()
    selected_ids = predictions["ADM2_CODE"].unique()
    first_by_district = (
        history_by_district[history_by_district["ADM2_CODE"].isin(selected_ids)]
        .groupby("ADM2_CODE")["date"].min()
    )
    if len(first_by_district) != len(selected_ids):
        raise ValueError("Training history missing for a selected district")
    # Lag creation gives some older runs extra early months for only a few IDs.
    # Start where every plotted district has observed training target values.
    history_dates = pd.date_range(first_by_district.max(), "2024-12-01", freq="MS")
    if len(history_dates) < 12:
        raise ValueError("Too little training history for timeline")

    summary_rows = []
    monthly_rows = []
    available_models = set(predictions["best_model"])
    if available_models - set(TIMELINE_MODEL_ORDER):
        raise ValueError(f"Unknown models: {sorted(available_models - set(TIMELINE_MODEL_ORDER))}")

    plt.style.use("seaborn-v0_8-darkgrid")
    for model_name in TIMELINE_MODEL_ORDER:
        selected = predictions.loc[predictions["best_model"] == model_name].copy()
        if selected.empty:
            print(f"No district selected {model_name}; no plot generated")
            continue

        district_ids = selected["ADM2_CODE"].unique()
        n_districts = len(district_ids)
        expected_pairs = pd.MultiIndex.from_product(
            [district_ids, expected_test_dates], names=["ADM2_CODE", "date"]
        )
        got_pairs = pd.MultiIndex.from_frame(selected[["ADM2_CODE", "date"]])
        if len(got_pairs) != len(expected_pairs) or not expected_pairs.isin(got_pairs).all():
            raise ValueError(f"{model_name}: incomplete 12-month test series")

        model_history = history_by_district.loc[
            history_by_district["ADM2_CODE"].isin(district_ids)
            & history_by_district["date"].isin(history_dates)
        ]
        expected_train_pairs = pd.MultiIndex.from_product(
            [district_ids, history_dates], names=["ADM2_CODE", "date"]
        )
        got_train_pairs = pd.MultiIndex.from_frame(model_history[["ADM2_CODE", "date"]])
        if len(got_train_pairs) != len(expected_train_pairs) or not expected_train_pairs.isin(got_train_pairs).all():
            raise ValueError(f"{model_name}: incomplete training series for selected districts")

        train_monthly = model_history.groupby("date")["target_cases"].mean()
        test_monthly = selected.groupby("date")["actual_cases"].mean()
        predicted_monthly = selected.groupby("date")["predicted_cases"].mean()
        if train_monthly.isna().any() or test_monthly.isna().any() or predicted_monthly.isna().any():
            raise ValueError(f"{model_name}: missing values in plot series")

        fig, ax = plt.subplots(figsize=(10.5, 4.5))
        ax.plot(train_monthly.index, train_monthly.values, label="Train", color="#1f77b4", linewidth=1.25)
        ax.plot(test_monthly.index, test_monthly.values, label="Test", color="#ff7f0e", linewidth=1.25)
        ax.plot(predicted_monthly.index, predicted_monthly.values, label="Predicted", color="#2ca02c", linewidth=1.25)
        ax.set_title(
            f"{model_name}: Train, Test and Predicted Monthly Malaria Cases\n"
            f"mean of {n_districts} districts that selected this model",
            fontsize=11, fontweight="bold",
        )
        ax.set_xlabel("Date")
        ax.set_ylabel("Mean malaria cases")
        ax.set_ylim(bottom=0)
        ax.set_xlim(history_dates[0] - pd.Timedelta(days=15), pd.Timestamp("2026-01-01"))
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        ax.tick_params(axis="x", labelrotation=0, labelsize=8)
        ax.legend(loc="upper right")
        fig.tight_layout()
        stem = f"train_test_predicted_{model_name}"
        fig.savefig(output_dir / f"{stem}.png", dpi=220)
        fig.savefig(output_dir / f"{stem}.pdf")
        plt.close(fig)

        for period, series in (
            ("Train", train_monthly), ("Test", test_monthly),
            ("Predicted", predicted_monthly),
        ):
            monthly_rows.extend(
                {"model": model_name, "district_count": n_districts,
                 "period": period, "date": date, "mean_cases": value}
                for date, value in series.items()
            )
        summary_rows.append({
            "model": model_name, "district_count": n_districts,
            "train_months": len(train_monthly), "test_months": len(test_monthly),
            "png": str(output_dir / f"{stem}.png"),
        })
        print(f"Saved {stem}: {n_districts} districts")

    pd.DataFrame(monthly_rows).to_csv(output_dir / "monthly_plot_values.csv", index=False)
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(output_dir / "plot_summary.csv", index=False)
    return summary



timeline_summary = make_plots(
    Path(OUTPUT_DIR),
    Path(OUTPUT_DIR) / 'train_test_predicted_by_model',
)
display(timeline_summary[['model', 'district_count', 'train_months', 'test_months']])


## SECTION Y1. TRAIN, TEST AND PREDICTED SERIES WITH 95% INTERVAL BY SELECTED MODEL

For each model, the lines are the monthly mean over the districts that selected that model: training target 2016-2024, observed 2025 test, and 2025 prediction. They are the same series as the Section Y timeline plots. The shaded band is the monthly mean of the district-level 95% interval bounds (`lower_95`, `upper_95`) of the 2025 predictions.

Interpretation notes:
- The band is the **average district 95% interval**, not a 95% confidence interval of the plotted mean. Between-district covariances are not available, so an interval for the mean cannot be derived; the band is wider than one would be.
- Interval methods differ by model (ARIMA/SARIMA/SARIMAX: statsmodels `conf_int`; XGBoost: prediction +/- 1.96 x in-sample residual SD; LSTM/GRU: MC dropout). Band widths are therefore not directly comparable across models.
- `coverage_95_2025` in `plot_summary_ci.csv` is the share of 2025 district-months whose observed (capped) value lies inside the interval; values well below 0.95 mean the interval is too narrow.

Outputs: `train_test_predicted_ci_by_model/` (PNG + PDF per model, `monthly_plot_values_ci.csv`, `plot_summary_ci.csv`).


In [ ]:
# ================================================================
# SECTION Y1. TRAIN, TEST AND PREDICTED SERIES WITH 95% INTERVAL
# BY SELECTED MODEL
# Same series and checks as the Section Y timeline plots
# (make_plots() in the Section Y timeline cell), plus the mean of the
# district-level 95% interval of the 2025 predictions as a shaded band.
# Reads only the CSVs exported above; it trains nothing.
# ================================================================
from pathlib import Path
import matplotlib.dates as mdates

Y1_DIR = Path(OUTPUT_DIR) / "train_test_predicted_ci_by_model"
Y1_DIR.mkdir(parents=True, exist_ok=True)
Y1_MODEL_ORDER = [
    "ARIMA", "SARIMA", "SARIMAX", "Univariate_LSTM",
    "Multivariate_LSTM", "GRU", "XGBoost",
]
_mc = f"MC dropout, {MC_DROPOUT_ITER} forecast paths, 2.5th-97.5th percentiles"
Y1_INTERVAL_METHOD = {
    "ARIMA": "statsmodels conf_int (95%), bounds clipped at 0",
    "SARIMA": "statsmodels conf_int (95%), bounds clipped at 0",
    "SARIMAX": "statsmodels conf_int (95%), bounds clipped at 0",
    "XGBoost": "prediction +/- 1.96 x in-sample residual SD, lower clipped at 0",
    "Univariate_LSTM": _mc,
    "Multivariate_LSTM": _mc,
    "GRU": _mc,
}

y1_pred = pd.read_csv(
    Path(OUTPUT_DIR) / "test_predictions_2025_best_model.csv",
    dtype={"ADM2_CODE": "string"}, parse_dates=["date"],
)
y1_hist = pd.read_csv(
    Path(OUTPUT_DIR) / "modeling_dataset_process_aligned_2016_2025.csv",
    dtype={"ADM2_CODE": "string"}, usecols=["ADM2_CODE", "date", "target_cases"],
    parse_dates=["date"],
)

# ---------------- checks ----------------
if y1_pred.empty or y1_hist.empty:
    raise ValueError("Prediction or modelling dataset is empty")
if y1_pred.duplicated(["ADM2_CODE", "date"]).any():
    raise ValueError("More than one 2025 prediction row for a district-month")
if y1_pred.groupby("ADM2_CODE")["best_model"].nunique().gt(1).any():
    raise ValueError("A district has more than one selected model")
if set(y1_pred["best_model"]) - set(Y1_MODEL_ORDER):
    raise ValueError(f"Unknown models: {sorted(set(y1_pred['best_model']) - set(Y1_MODEL_ORDER))}")
y1_test_dates = pd.date_range("2025-01-01", "2025-12-01", freq="MS")
if set(y1_pred["date"]) != set(y1_test_dates):
    raise ValueError("2025 predictions must cover exactly Jan-Dec 2025")
y1_bounds_ok = y1_pred[["lower_95", "upper_95"]].notna().all(axis=1)
if not y1_bounds_ok.all():
    print(f"Warning: {int((~y1_bounds_ok).sum())} prediction rows have no 95% interval; "
          "the band averages the rows that have one")

# Training history: the same window for every model, starting at the first
# month in which every selected district has an observed training value.
y1_hist = y1_hist.loc[y1_hist["date"] < y1_test_dates[0]]
y1_hist = y1_hist.groupby(["ADM2_CODE", "date"], as_index=False)["target_cases"].sum()
y1_ids = y1_pred["ADM2_CODE"].unique()
y1_first = y1_hist[y1_hist["ADM2_CODE"].isin(y1_ids)].groupby("ADM2_CODE")["date"].min()
if len(y1_first) != len(y1_ids):
    raise ValueError("Training history missing for a selected district")
y1_train_dates = pd.date_range(y1_first.max(), "2024-12-01", freq="MS")

# ---------------- plots ----------------
plt.style.use("seaborn-v0_8-darkgrid")
y1_monthly_rows, y1_summary_rows = [], []
for model_name in Y1_MODEL_ORDER:
    sel = y1_pred.loc[y1_pred["best_model"] == model_name]
    if sel.empty:
        print(f"No district selected {model_name}; no plot generated")
        continue
    ids = sel["ADM2_CODE"].unique()
    n_districts = len(ids)
    if len(sel) != n_districts * len(y1_test_dates):
        raise ValueError(f"{model_name}: incomplete 12-month test series")
    hist = y1_hist.loc[y1_hist["ADM2_CODE"].isin(ids) & y1_hist["date"].isin(y1_train_dates)]
    if len(hist) != n_districts * len(y1_train_dates):
        raise ValueError(f"{model_name}: incomplete training series for selected districts")

    train_m = hist.groupby("date")["target_cases"].mean()
    by_month = sel.groupby("date")
    test_m = by_month["actual_cases"].mean()
    pred_m = by_month["predicted_cases"].mean()
    lower_m = by_month["lower_95"].mean()
    upper_m = by_month["upper_95"].mean()
    inside = sel.loc[y1_bounds_ok.loc[sel.index]]
    coverage = float(((inside["actual_cases"] >= inside["lower_95"])
                      & (inside["actual_cases"] <= inside["upper_95"])).mean())

    fig, ax = plt.subplots(figsize=(10.5, 4.5))
    ax.plot(train_m.index, train_m.values, label="Train", color="#1f77b4", linewidth=1.25)
    ax.plot(test_m.index, test_m.values, label="Test", color="#ff7f0e", linewidth=1.25)
    ax.plot(pred_m.index, pred_m.values, label="Predicted", color="#2ca02c", linewidth=1.25)
    ax.fill_between(pred_m.index, lower_m.values, upper_m.values, color="#2ca02c",
                    alpha=0.2, linewidth=0, label="Predicted 95% interval")
    ax.set_title(
        f"{model_name}: Train, Test and Predicted Monthly Malaria Cases\n"
        f"mean of {n_districts} districts that selected this model",
        fontsize=11, fontweight="bold",
    )
    ax.set_xlabel("Date")
    ax.set_ylabel("Mean malaria cases")
    # Headroom so the one-row legend never covers the data.
    y_top = max(train_m.max(), test_m.max(), pred_m.max(), upper_m.max())
    ax.set_ylim(0, y_top * 1.2 if y_top > 0 else 1)
    ax.set_xlim(y1_train_dates[0] - pd.Timedelta(days=15), pd.Timestamp("2026-01-01"))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.tick_params(axis="x", labelrotation=0, labelsize=8)
    ax.legend(loc="upper left", ncol=4, frameon=True, framealpha=0.9, fontsize=9)
    fig.tight_layout()
    stem = f"train_test_predicted_ci_{model_name}"
    fig.savefig(Y1_DIR / f"{stem}.png", dpi=220)
    fig.savefig(Y1_DIR / f"{stem}.pdf")
    plt.show()
    plt.close(fig)

    for period, series in (("Train", train_m), ("Test", test_m), ("Predicted", pred_m)):
        for date, value in series.items():
            y1_monthly_rows.append({
                "model": model_name, "district_count": n_districts, "period": period,
                "date": date, "mean_cases": value,
                "mean_lower_95": lower_m[date] if period == "Predicted" else np.nan,
                "mean_upper_95": upper_m[date] if period == "Predicted" else np.nan,
            })
    y1_summary_rows.append({
        "model": model_name, "district_count": n_districts,
        "interval_method": Y1_INTERVAL_METHOD[model_name],
        "coverage_95_2025": round(coverage, 3),
        "png": str(Y1_DIR / f"{stem}.png"),
    })

y1_monthly = pd.DataFrame(y1_monthly_rows)
y1_summary = pd.DataFrame(y1_summary_rows)
y1_monthly.to_csv(Y1_DIR / "monthly_plot_values_ci.csv", index=False)
y1_summary.to_csv(Y1_DIR / "plot_summary_ci.csv", index=False)
display(y1_summary[["model", "district_count", "interval_method", "coverage_95_2025"]])
print("Section Y1 outputs saved to:", Y1_DIR)


## SECTION Z. RISK AREA IDENTIFICATION RESULTS: HOTSPOT MAP-READY OUTPUTS

This section prepares hotspot map outputs using predicted annual 2026 cases and the risk classification rule: high risk if forecasted 2026 cases exceed the recent five-year annual median for the same district.

In [ ]:
# ================================================================
# SECTION Z. RISK AREA IDENTIFICATION RESULTS: HOTSPOT MAP-READY OUTPUTS
# ================================================================
risk_map_df = risk_df.copy()

# Attach coordinates if available from climate data.
if lat_col and lon_col:
    coord_df = climate_plot_df.groupby("ADM2_CODE", as_index=False)[[lat_col, lon_col]].mean()
    risk_map_df = risk_map_df.merge(coord_df, on="ADM2_CODE", how="left")
    plt.figure(figsize=(7, 9))
    risk_num = risk_map_df["risk_level"].map({"Low risk": 0, "High risk": 1})
    sc = plt.scatter(risk_map_df[lon_col], risk_map_df[lat_col], c=risk_num, s=20)
    plt.colorbar(sc, ticks=[0, 1], label="Risk level: 0=Low, 1=High")
    plt.title(f"Predicted Malaria Hotspot Districts, {FORECAST_YEAR}")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/hotspot_map_predicted_risk_scatter_{FORECAST_YEAR}.png", dpi=300)
    plt.show()
else:
    print("Latitude/longitude were not available. Exporting map-ready risk table only.")

risk_map_summary = risk_map_df.groupby("risk_level", as_index=False).size().rename(columns={"size": "district_count"})
display(risk_map_summary)
display(risk_map_df.sort_values("forecast_cases_2026", ascending=False).head(20))

risk_map_df.to_csv(f"{OUTPUT_DIR}/hotspot_map_ready_risk_districts_{FORECAST_YEAR}.csv", index=False)
risk_map_summary.to_csv(f"{OUTPUT_DIR}/hotspot_risk_summary_{FORECAST_YEAR}.csv", index=False)
mark_section_checkpoint("Z_hotspot_maps")

## SECTION AA. OPTIONAL GEOAI SPATIAL ANALYSIS AND MAP-READY EXPORT

This section joins district-level risk results to an administrative boundary file. GeoPandas performs the reproducible vector join. When `geoai-py` is enabled, OpenGeoAI is additionally used to inspect and visualize the exported vector layer. A local boundary path or a direct-download URL may be supplied; the administrative source must be cited separately in the thesis.


In [ ]:
# ================================================================
# OPTIONAL GEOAI / GEOPANDAS SPATIAL OUTPUT
# ================================================================
DISTRICT_BOUNDARY_PATH = ""  # e.g., "/content/drive/MyDrive/data/thailand_adm2.geojson"
DISTRICT_BOUNDARY_URL = ""   # optional direct URL; used only when GeoAI is enabled
BOUNDARY_JOIN_FIELD = "ADM2_CODE"  # here: a province boundary field holding the 2-digit province code

boundary_path = DISTRICT_BOUNDARY_PATH
if not boundary_path and DISTRICT_BOUNDARY_URL:
    if not GEOAI_AVAILABLE:
        raise ImportError("Enable INSTALL_OPEN_GEOAI before downloading a boundary file through GeoAI.")
    boundary_path = geoai.download_file(DISTRICT_BOUNDARY_URL)

if boundary_path and os.path.exists(boundary_path):
    districts_gdf = gpd.read_file(boundary_path)
    if BOUNDARY_JOIN_FIELD not in districts_gdf.columns:
        raise KeyError(f"Boundary join field not found: {BOUNDARY_JOIN_FIELD}")

    districts_gdf[BOUNDARY_JOIN_FIELD] = districts_gdf[BOUNDARY_JOIN_FIELD].astype(str)
    spatial_risk = risk_df.copy()
    spatial_risk["ADM2_CODE"] = spatial_risk["ADM2_CODE"].astype(str)

    risk_map_gdf = districts_gdf.merge(
        spatial_risk,
        left_on=BOUNDARY_JOIN_FIELD,
        right_on="ADM2_CODE",
        how="left",
        validate="one_to_one",
    )
    geojson_output = os.path.join(OUTPUT_DIR, "malaria_risk_2026_geoai.geojson")
    gpkg_output = os.path.join(OUTPUT_DIR, "malaria_risk_2026_geoai.gpkg")
    risk_map_gdf.to_file(geojson_output, driver="GeoJSON")
    risk_map_gdf.to_file(gpkg_output, layer="malaria_risk_2026", driver="GPKG")

    ax = risk_map_gdf.plot(
        column="risk_level",
        categorical=True,
        legend=True,
        cmap="RdYlGn_r",
        edgecolor="white",
        linewidth=0.15,
        figsize=(10, 14),
        missing_kwds={"color": "lightgrey", "label": "No matched result"},
    )
    ax.set_title("District-level malaria risk classification, Thailand, 2026")
    ax.set_axis_off()
    plt.tight_layout()
    save_current_figure("malaria_risk_2026_geoai_map")

    if GEOAI_AVAILABLE:
        geoai.print_vector_info(geojson_output)
        geoai.view_vector(geojson_output, basemap=True)
else:
    print("Spatial join skipped. Provide DISTRICT_BOUNDARY_PATH or DISTRICT_BOUNDARY_URL to enable it.")

mark_section_checkpoint("AA_geoai_export")